# FlowGraphFormer — 03_Big_Modeling.ipynb
**An Automated Real-Time Intrusion Detection and Prevention System Using Machine Learning**

This notebook implements and rigorously compares 7 models on the preprocessed dataset produced by
`01_Preprocessing_local.ipynb`:

1. CatBoost
2. FT-Transformer
3. TabM (ICLR 2025, simplified faithful reproduction)
4. TabGT (2024, simplified faithful reproduction)
5. GNNFormer (2024, simplified faithful reproduction)
6. GraphTransformer (feature-graph transformer)
7. TabPFN-2.5 (2025)
8. Hybrid / Stacked Ensemble (2–3 best models)

It also produces: 5–10 fold cross-validation, data-leakage audits, paired t-test / McNemar's test
significance tables, and SHAP / Permutation Importance / LIME explainability (XAI) figures.

**Every cell is independent** — each one only needs Cells 1–8 (setup, config, data, helpers) to have
been run once. Models do **not** depend on each other, so you can run them in any order or re-run a
single one after a crash without redoing the rest.

> Note on TabM / TabGT / GNNFormer / GraphTransformer: these are very recent research architectures
> that do not (yet) have a stable, pip-installable, Colab-safe official package. To keep this
> notebook 100% runnable, they are implemented here as compact, paper-faithful PyTorch
> re-implementations of the core idea of each method (see the comment block at the top of each
> model cell for the exact simplification made). If your supervisor specifically wants the authors'
> official code, that should be cited as a limitation / future work in your report.


In [ ]:
# ==============================================================================
# Cell 1 : Mount Drive & Project Setup
# ==============================================================================

# ------------------------------------------------------------------
# Auto-detects whether this notebook is running on Google Colab or on
# your local machine (mirrors 01_Preprocessing_local.ipynb Cell 0 logic).
# find_spec("google.colab") raises ModuleNotFoundError locally rather
# than returning None, so we use try/except instead of a None-check.
# ------------------------------------------------------------------

from pathlib import Path
import sys

try:
    from google.colab import drive

    drive.mount("/content/drive")

    RUN_ENVIRONMENT = "colab"

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/ML_Project/FlowGraphFormer"
    )

except ModuleNotFoundError:

    RUN_ENVIRONMENT = "local"

    # Uses the current working directory (wherever you launched this
    # notebook from on your ASUS laptop) instead of a hardcoded path
    PROJECT_ROOT = Path.cwd()

    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 80)
print("FlowGraphFormer")
print("Research Notebook : 03_Big_Modeling")
print("=" * 80)
print("Run Environment :", RUN_ENVIRONMENT)
print("Project Root    :", PROJECT_ROOT)

# ==============================================================================
# Memory Management Helpers (same pattern as 01_Preprocessing_local.ipynb)
# ==============================================================================
import gc

def print_ram_usage(label=""):
    """Print current process RAM usage, if psutil is available."""
    try:
        import psutil
        import os as _os
        process = psutil.Process(_os.getpid())
        rss_gb = process.memory_info().rss / 1024 ** 3
        total_gb = psutil.virtual_memory().total / 1024 ** 3
        print(f"RAM Usage - {label} : {rss_gb:.2f} GB / {total_gb:.2f} GB")
    except ImportError:
        pass


def free_memory(*names, scope):
    """Delete the given variable names from `scope` (usually globals()) if present."""
    for name in names:
        if name in scope:
            del scope[name]
    gc.collect()


def downcast_dtypes(df, verbose=True):
    """Downcast numeric columns in-place to the smallest safe dtype."""
    start_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    end_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    if verbose:
        print(f"Downcast dtypes : {start_mb:,.1f} MB -> {end_mb:,.1f} MB "
              f"({(1 - end_mb / start_mb) * 100:.1f}% reduction)")
    return df


def cleanup_torch():
    """Frees Python-side RAM and, if a GPU is in use, releases its cached
    memory pool back so it doesn't accumulate fold-over-fold / model-over-model.
    Handles NVIDIA/AMD (cuda), Intel (xpu), and Apple (mps) backends."""
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        if hasattr(torch, "xpu") and torch.xpu.is_available():
            torch.xpu.empty_cache()

        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

    except ImportError:
        pass
    except Exception:
        # empty_cache()/ipc_collect() can be flaky on some driver versions --
        # never let cleanup itself crash a training run
        pass


In [ ]:
# ==============================================================================
# Cell 2 : Install Required Packages
# ==============================================================================
#
# All packages needed by every model / XAI / stats cell below are installed
# ONCE here so that every later cell can be re-run independently without
# re-installing anything.
# ==============================================================================

import sys
import subprocess
import importlib

print("=" * 80)
print("Installing / Verifying Packages")
print("=" * 80)

REQUIRED_PACKAGES = {
    "catboost": "catboost",
    "torch": "torch",
    "tabpfn": "tabpfn",
    "shap": "shap",
    "lime": "lime",
    "statsmodels": "statsmodels",
    "mlxtend": "mlxtend",
    "scikit_posthocs": "scikit-posthocs",
}

missing_packages = []

for module, package in REQUIRED_PACKAGES.items():

    try:

        importlib.import_module(module)

    except ImportError:

        missing_packages.append(package)

if missing_packages:

    print("\nInstalling :", missing_packages, "\n")

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"]
        + missing_packages
    )

    print("\nInstallation completed.\n")

else:

    print("\nAll required packages already installed.\n")

print("=" * 80)
print("Done")
print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 3 : Project Configuration (Directories)
# ==============================================================================
# Mirrors the directory layout created in 01_Preprocessing_local.ipynb so that this
# notebook reads exactly what that notebook saved.
# ==============================================================================

from pathlib import Path


# ==============================================================================
# Dataset Directories (produced by 01_Preprocessing_local.ipynb)
# ==============================================================================

DATASET_DIR = PROJECT_ROOT / "dataset"

PROCESSED_DATASET_DIR = DATASET_DIR / "processed" / "final"

# ==============================================================================
# Modeling Reports (new, created by THIS notebook)
# ==============================================================================

REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_REPORT_DIR = REPORT_DIR / "modeling"

FIGURE_DIR = MODEL_REPORT_DIR / "figures"

TABLE_DIR = MODEL_REPORT_DIR / "tables"

XAI_DIR = MODEL_REPORT_DIR / "xai"

STATS_DIR = MODEL_REPORT_DIR / "stats"

PRED_DIR = MODEL_REPORT_DIR / "predictions"

# ==============================================================================
# Saved Model Objects
# ==============================================================================

MODEL_DIR = PROJECT_ROOT / "models"

TRAINED_MODEL_DIR = MODEL_DIR / "trained"

ENCODER_DIR = MODEL_DIR / "encoders"

# ==============================================================================
# Create Directories
# ==============================================================================

DIRECTORIES = [
    PROJECT_ROOT,
    DATASET_DIR,
    PROCESSED_DATASET_DIR,
    REPORT_DIR,
    MODEL_REPORT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    XAI_DIR,
    STATS_DIR,
    PRED_DIR,
    MODEL_DIR,
    TRAINED_MODEL_DIR,
    ENCODER_DIR,
]

for directory in DIRECTORIES:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("=" * 80)
print("Directories Ready")
print("=" * 80)

for directory in DIRECTORIES:

    print(directory)


In [ ]:
# ==============================================================================
# Cell 4 : MASTER CONTROL CELL
# ==============================================================================
# Every important variable used anywhere in this notebook is controlled here.
# Change values in the form fields (Colab renders these as widgets) and
# re-run — no need to touch any other cell.
# ==============================================================================

# ------------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------------
RANDOM_SEED = 42  #@param {type:"integer"}

# ------------------------------------------------------------------
# Dataset files (as saved by 01_Preprocessing_local.ipynb)
# ------------------------------------------------------------------
# NOTE: 01_Preprocessing_local.ipynb now saves the large SMOTE/scaled frames
# as Snappy-compressed Parquet, not CSV (see its Cell 12/13/17). Only
# the small y_validation / y_test label files are still plain CSV.
TRAIN_FEATURES_FILE = "X_train_smote.parquet"        #@param {type:"string"}
VAL_FEATURES_FILE   = "X_validation_scaled.parquet"  #@param {type:"string"}
TEST_FEATURES_FILE  = "X_test_scaled.parquet"        #@param {type:"string"}

TRAIN_LABELS_FILE = "y_train_smote.parquet"    #@param {type:"string"}
VAL_LABELS_FILE   = "y_validation.csv"   #@param {type:"string"}
TEST_LABELS_FILE  = "y_test.csv"         #@param {type:"string"}

TARGET_COLUMN = "Label"  #@param {type:"string"}

# ------------------------------------------------------------------
# Cross-Validation
# ------------------------------------------------------------------
N_FOLDS = 10                  #@param {type:"slider", min:5, max:10, step:1}
CV_ON = "train_only"          #@param ["train+val", "train_only"]

# RUN_VALIDATION = False skips CV / validation scoring ENTIRELY for every
# model cell below -- each model just does a single final fit on the full
# CV pool and reports the held-out test set result. Useful for a fast
# end-to-end smoke run before committing to a full 7-model x N-fold pass.
RUN_VALIDATION = True          #@param {type:"boolean"}

# USE_KFOLD = True  -> CV runs as N_FOLDS stratified/plain folds (as before)
# USE_KFOLD = False -> CV runs as ONE train/val split (size VAL_SPLIT_SIZE)
# Ignored entirely if RUN_VALIDATION = False.
USE_KFOLD = True               #@param {type:"boolean"}
VAL_SPLIT_SIZE = 0.2           #@param {type:"number"}
# NOTE (fixed): "train+val" mixes the SMOTE-balanced X_train with the
# naturally-imbalanced X_val into a single CV pool. That contaminates
# every CV fold with synthetic rows and lets the model tune itself to a
# balanced world it will never see at real test time -- this is why
# test/val results look much worse than CV results (great CV score,
# bad BENIGN-heavy real-world score). Keep "train_only" unless you
# deliberately want val folded into CV.
STRATIFIED = True            #@param {type:"boolean"}

# ------------------------------------------------------------------
# Class imbalance handling for THIS notebook's own CV folds
# TRAIN_FEATURES_FILE (X_train_smote.parquet) is ALREADY SMOTE-balanced
# by 01_Preprocessing_local.ipynb. Setting this True would apply SMOTE a
# SECOND time on top of already-synthetic data every fold -- keep this
# False unless TRAIN_FEATURES_FILE is repointed at a non-SMOTE file.
# ------------------------------------------------------------------
USE_SMOTE_PER_FOLD = False  #@param {type:"boolean"}

# ------------------------------------------------------------------
# Full Dataset vs Smoke Test / Subsample control
# ------------------------------------------------------------------
USE_FULL_DATASET = True  #@param {type:"boolean"}

SMOKE_TEST = False         #@param {type:"boolean"}
SMOKE_TEST_ROWS = 1000    #@param {type:"integer"}

# ------------------------------------------------------------------
# Compute budget / GPU backend (manual, cross-vendor)
# ------------------------------------------------------------------
# "auto"  -> picks the best available backend automatically, in order
#            NVIDIA/AMD (cuda) -> Intel (xpu) -> Apple (mps) -> cpu
# "cuda"  -> NVIDIA GPUs, AND AMD GPUs running a ROCm build of PyTorch
#            (ROCm exposes itself through the SAME torch.cuda.* API, so
#            no separate "rocm" option is needed here)
# "xpu"   -> Intel GPUs (Arc / Data Center GPU / integrated) via
#            torch's native XPU backend (torch>=2.4) or the
#            intel-extension-for-pytorch package on older torch versions
# "mps"   -> Apple Silicon GPUs (M1/M2/M3/...), included for completeness
# "cpu"   -> force CPU regardless of what GPUs are detected
DEVICE_PREFERENCE = "auto"   #@param ["auto", "cuda", "xpu", "mps", "cpu"]

# Which physical GPU to use when more than one is present (0-indexed).
# Ignored for "mps" (Apple only ever exposes a single logical GPU) and
# for "cpu". Set this manually after checking the "Detected Devices"
# printout below.
GPU_INDEX = 0                #@param {type:"integer"}

NN_EPOCHS = 20               #@param {type:"integer"}
NN_BATCH_SIZE = 2048         #@param {type:"integer"}
NN_LEARNING_RATE = 1e-3     #@param {type:"number"}
NN_PATIENCE = 15             #@param {type:"integer"}

# Keras-style per-epoch printing ("Epoch 001/100 - loss: 0.1234") from
# train_torch_classifier(). Turn off for a quieter log on long full-dataset
# runs across 5-10 folds x 5 neural models.
NN_VERBOSE = True            #@param {type:"boolean"}

# NOTE: NN_BATCH_SIZE is now fully user-controlled from this cell, for
# both full-dataset and smoke-test/subsample runs. It is NO LONGER
# force-overridden when USE_FULL_DATASET=True -- whatever you set above
# is what gets used. Cell 10's train_torch_classifier() still clamps it
# DOWN automatically per-fold (effective_batch_size = min(NN_BATCH_SIZE,
# len(X_tr)//4)) purely as a safety net so tiny folds (e.g. SMOKE_TEST)
# never end up with a single unstable batch -- it will never raise it
# above what you set here.
if NN_BATCH_SIZE < 8:

    print(f"NOTE: NN_BATCH_SIZE={NN_BATCH_SIZE} is very small -- "
          f"training will be slow and batch statistics may be unstable. "
          f"1024 is a reasonable default for full-dataset runs.")

# ------------------------------------------------------------------
# TabPFN — subsample large IDS data down to TabPFN's supported size
# ------------------------------------------------------------------
TABPFN_MAX_TRAIN_ROWS = 10000  #@param {type:"integer"}
TABPFN_MAX_FEATURES  = 100   #@param {type:"integer"}

# ------------------------------------------------------------------
# Hybrid / ensemble
# ------------------------------------------------------------------
HYBRID_AUTO_SELECT = False        #@param {type:"boolean"}
HYBRID_TOP_N = 3                 #@param {type:"slider", min:2, max:3, step:1}
HYBRID_SELECT_METRIC = "roc_auc_ovr"  #@param ["f1_macro", "accuracy", "mcc", "roc_auc_ovr"]

# Used ONLY if HYBRID_AUTO_SELECT = False (manual override)
HYBRID_MEMBERS_MANUAL = ["gnnformer", "fttransformer", "graphtransformer"]  #@param {type:"raw"}

HYBRID_STRATEGY = "stacking"  #@param ["stacking", "soft_voting"]

# ------------------------------------------------------------------
# XAI
# ------------------------------------------------------------------
XAI_SAMPLE_SIZE = 500             #@param {type:"integer"}
XAI_TOP_FEATURES = 25            #@param {type:"integer"}
LIME_INSTANCE_INDEX = 0          #@param {type:"integer"}

# Which models to explain -- SHAP / Permutation Importance / LIME cells below
# all loop over this list, generating a full set of per-model XAI figures for
# every base model AND the hybrid ensemble (predictions combined the same way
# Cell 17 combines them: stacking meta-model, or soft-voting fallback).
XAI_MODELS = [ "catboost", "fttransformer", "tabm", "tabgt", "gnnformer", "graphtransformer", "tabpfn", "hybrid" ] #@param {type:"raw"}

# Permutation Importance -- lower than sklearn's usual default (10) since
# this now runs across 8 models instead of 1; raise it back up for a final
# thesis run if runtime allows.
PERM_N_REPEATS = 5  #@param {type:"integer"}

# LIME -- number of local perturbed samples per explained instance. LIME's
# own default is 5000; lowered here for the same reason (8 models x this
# many predict_fn calls each).
LIME_NUM_SAMPLES = 500  #@param {type:"integer"}

# ------------------------------------------------------------------
# Figures
# ------------------------------------------------------------------
FIGURE_DPI = 300      #@param {type:"integer"}
FIGURE_FORMAT = "png" #@param ["png", "pdf", "svg"]

# ------------------------------------------------------------------
# Bundle everything so every later cell reads from ONE dict (no
# scattered globals, easy to log / save with the results)
# ------------------------------------------------------------------
CONFIG = {
    "random_seed": RANDOM_SEED,
    "n_folds": N_FOLDS,
    "cv_on": CV_ON,
    "run_validation": RUN_VALIDATION,
    "use_kfold": USE_KFOLD,
    "val_split_size": VAL_SPLIT_SIZE,
    "stratified": STRATIFIED,
    "use_smote_per_fold": USE_SMOTE_PER_FOLD,
    "use_full_dataset": USE_FULL_DATASET,
    "smoke_test": SMOKE_TEST,
    "smoke_test_rows": SMOKE_TEST_ROWS,
    "device_preference": DEVICE_PREFERENCE,
    "gpu_index": GPU_INDEX,
    "nn_epochs": NN_EPOCHS,
    "nn_batch_size": NN_BATCH_SIZE,
    "nn_learning_rate": NN_LEARNING_RATE,
    "nn_patience": NN_PATIENCE,
    "nn_verbose": NN_VERBOSE,
    "tabpfn_max_train_rows": TABPFN_MAX_TRAIN_ROWS,
    "tabpfn_max_features": TABPFN_MAX_FEATURES,
    "hybrid_auto_select": HYBRID_AUTO_SELECT,
    "hybrid_top_n": HYBRID_TOP_N,
    "hybrid_select_metric": HYBRID_SELECT_METRIC,
    "hybrid_members_manual": HYBRID_MEMBERS_MANUAL,
    "hybrid_strategy": HYBRID_STRATEGY,
    "xai_sample_size": XAI_SAMPLE_SIZE,
    "xai_top_features": XAI_TOP_FEATURES,
    "lime_instance_index": LIME_INSTANCE_INDEX,
    "xai_models": XAI_MODELS,
    "perm_n_repeats": PERM_N_REPEATS,
    "lime_num_samples": LIME_NUM_SAMPLES,
    "figure_dpi": FIGURE_DPI,
    "figure_format": FIGURE_FORMAT,
}

import random
import numpy as np

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

try:
    import torch
    torch.manual_seed(CONFIG["random_seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CONFIG["random_seed"])
except ImportError:
    pass

print("=" * 80)
print("CONFIG")
print("=" * 80)

for key, value in CONFIG.items():

    print(f"{key:<25}: {value}")

import json as _json

with open(MODEL_REPORT_DIR / "run_config.json", "w") as f:

    _json.dump(CONFIG, f, indent=4)

print()
print("Saved:", MODEL_REPORT_DIR / "run_config.json")

# ==============================================================================
# GPU / Compute Backend Detection (manual-select support: NVIDIA, AMD, Intel)
# ==============================================================================
# Run this after setting DEVICE_PREFERENCE / GPU_INDEX above. It only
# REPORTS what's available -- Cell 8's get_device() (used by every model
# cell) is what actually reads CONFIG["device_preference"] / GPU_INDEX and
# selects the device, so re-running THIS cell after changing the dropdown
# is enough; no other cell needs to change.

def _detect_available_devices():
    import torch

    report = {"cuda": [], "xpu": [], "mps": False}

    # ---- NVIDIA / AMD (ROCm) -- both surface through torch.cuda.* ----
    try:
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                name = torch.cuda.get_device_name(i)
                # crude vendor guess purely for the printout; ROCm builds
                # of torch report AMD GPU names here (e.g. "AMD Radeon ...",
                # "Instinct MI..."), CUDA builds report NVIDIA names.
                vendor = "AMD (ROCm)" if "amd" in name.lower() or "radeon" in name.lower() or "instinct" in name.lower() else "NVIDIA (CUDA)"
                report["cuda"].append(f"cuda:{i} - {name} [{vendor}]")
    except Exception as e:
        print(f"  (CUDA/ROCm detection skipped: {e})")

    # ---- Intel GPUs ----
    # torch>=2.4 ships a native `torch.xpu` backend. On older torch
    # installs, `intel_extension_for_pytorch` (ipex) registers the same
    # `xpu` device type as a side effect of being imported.
    try:
        if not hasattr(torch, "xpu"):
            import intel_extension_for_pytorch as ipex  # noqa: F401
        if hasattr(torch, "xpu") and torch.xpu.is_available():
            for i in range(torch.xpu.device_count()):
                name = torch.xpu.get_device_name(i)
                report["xpu"].append(f"xpu:{i} - {name} [Intel]")
    except ImportError:
        pass
    except Exception as e:
        print(f"  (Intel XPU detection skipped: {e})")

    # ---- Apple Silicon ----
    try:
        report["mps"] = torch.backends.mps.is_available()
    except Exception:
        pass

    return report


try:
    import torch

    _devices = _detect_available_devices()

    print()
    print("=" * 80)
    print("Detected Compute Devices")
    print("=" * 80)

    if _devices["cuda"]:
        for line in _devices["cuda"]:
            print(" ", line)
    else:
        print("  No NVIDIA/AMD (CUDA/ROCm) GPU detected")

    if _devices["xpu"]:
        for line in _devices["xpu"]:
            print(" ", line)
    else:
        print("  No Intel (XPU) GPU detected")

    print(f"  Apple (MPS) available : {_devices['mps']}")

    print()
    print(f"DEVICE_PREFERENCE = \"{CONFIG['device_preference']}\"   GPU_INDEX = {CONFIG['gpu_index']}")
    print("(Actual device selection happens per-model via Cell 8's get_device())")
    print("=" * 80)

except ImportError:
    print("PyTorch not installed yet -- run Cell 2 first, then re-run this cell.")


In [ ]:
# ==============================================================================
# Cell 5 : Load Preprocessed Dataset
# ==============================================================================
# Loads exactly what 01_Preprocessing_local.ipynb produced. Nothing is refit here —
# the scaler / encoder / SMOTE were already fit ONLY on the training split
# inside the preprocessing notebook, so re-loading these CSVs is leakage-safe.
# ==============================================================================

import pandas as pd
import numpy as np
import joblib
from tqdm.auto import tqdm

print("=" * 80)
print("Loading Preprocessed Dataset")
print("=" * 80)

# ------------------------------------------------------------------
# Each file is loaded one at a time, with its name and a progress bar,
# so you can see exactly which file is being read and confirm it's the
# right one before moving to the next.
# ------------------------------------------------------------------
FILES_TO_LOAD = [
    ("X_train", PROCESSED_DATASET_DIR / TRAIN_FEATURES_FILE),
    ("X_val",   PROCESSED_DATASET_DIR / VAL_FEATURES_FILE),
    ("X_test",  PROCESSED_DATASET_DIR / TEST_FEATURES_FILE),
    ("y_train", PROCESSED_DATASET_DIR / TRAIN_LABELS_FILE),
    ("y_val",   PROCESSED_DATASET_DIR / VAL_LABELS_FILE),
    ("y_test",  PROCESSED_DATASET_DIR / TEST_LABELS_FILE),
]

loaded = {}

for name, filepath in tqdm(FILES_TO_LOAD, desc="Loading Files"):

    print(f"\nLoading : {name:<8} <- {filepath.name}")

    if not filepath.exists():

        raise FileNotFoundError(f"Missing file expected at: {filepath}")

    # Read Parquet or CSV depending on the extension -- 01_Preprocessing
    # saves the large SMOTE/scaled frames as Parquet and the small
    # label files (y_val, y_test) as CSV.
    if filepath.suffix == ".parquet":

        df = pd.read_parquet(filepath, engine="pyarrow")

    else:

        df = pd.read_csv(filepath)

    downcast_dtypes(df, verbose=False)

    loaded[name] = df

    print(f"  Path   : {filepath}")
    print(f"  Shape  : {df.shape}")

X_train = loaded["X_train"]
X_val   = loaded["X_val"]
X_test  = loaded["X_test"]

y_train = loaded["y_train"].iloc[:, 0].values
y_val   = loaded["y_val"].iloc[:, 0].values
y_test  = loaded["y_test"].iloc[:, 0].values

# ------------------------------------------------------------------
# Label encoder
# ------------------------------------------------------------------
print(f"\nLoading : label_encoder <- {ENCODER_DIR / 'label_encoder.pkl'}")

label_encoder = joblib.load(ENCODER_DIR / "label_encoder.pkl")

CLASS_NAMES = list(label_encoder.classes_)
N_CLASSES = len(CLASS_NAMES)
FEATURE_NAMES = list(X_train.columns)
N_FEATURES = len(FEATURE_NAMES)

# ==============================================================================
# X / y Row-Count Consistency Check (fails fast, with a clear diagnosis)
# ==============================================================================
print()
print("=" * 80)
print("X / y Row-Count Consistency Check")
print("=" * 80)

_pairs = [
    ("train", X_train, y_train, TRAIN_FEATURES_FILE, TRAIN_LABELS_FILE),
    ("val",   X_val,   y_val,   VAL_FEATURES_FILE,   VAL_LABELS_FILE),
    ("test",  X_test,  y_test,  TEST_FEATURES_FILE,  TEST_LABELS_FILE),
]

_mismatch_found = False

for split_name, X_split, y_split, x_file, y_file in _pairs:

    match = len(X_split) == len(y_split)

    status = "PASS" if match else "FAIL"

    print(f"  [{split_name:<5}] {x_file:<25} ({len(X_split):>9,}) "
          f"vs {y_file:<20} ({len(y_split):>9,})  ->  {status}")

    if not match:

        _mismatch_found = True

        ratio = len(X_split) / len(y_split) if len(y_split) else float("nan")

        print(f"           Ratio (X/y) : {ratio:.3f}")

if _mismatch_found:

    raise ValueError(
        "\nX/y row counts do not match for one or more splits (see above).\n"
        "This almost always means X_*_scaled.csv and y_*.csv were saved from\n"
        "DIFFERENT stages of 01_Preprocessing_local.ipynb (e.g. one is post-SMOTE,\n"
        "the other pre-SMOTE), or from two different preprocessing runs.\n"
        "Fix this in 01_Preprocessing_local.ipynb before continuing:\n"
        "  - Make sure X_train_scaled.csv and y_train.csv are saved in the\n"
        "    SAME cell, right after the SAME train/val/test split, and that\n"
        "    neither one is re-saved later after SMOTE overwrites X_train.\n"
        "  - If SMOTE output should be used for training, save it separately\n"
        "    as X_train_smote.csv / y_train_smote.csv (as your preprocessing\n"
        "    notebook already does), and point TRAIN_FEATURES_FILE / \n"
        "    TRAIN_LABELS_FILE in Cell 4 at THOSE files instead if that's\n"
        "    the intended training data."
    )

print("\nX/y row-count check: ALL PASS (train, val, test all aligned)")

print()
print("=" * 80)
print("Dataset Loaded")
print("=" * 80)
print(f"{'X_train':<10}: {X_train.shape}")
print(f"{'X_val':<10}: {X_val.shape}")
print(f"{'X_test':<10}: {X_test.shape}")
print()
print("Classes    :", CLASS_NAMES)
print("N Classes  :", N_CLASSES)
print("N Features :", N_FEATURES)

# ==============================================================================
# CV Pool — either (train) or (train + val) depending on CONFIG["cv_on"]
# The held-out TEST set is NEVER used inside cross-validation, and is only
# ever touched once per model, at final evaluation. This is the main
# data-leakage guard for this notebook.
# ==============================================================================

print()
print("=" * 80)
print(f"Building CV Pool (mode: {CONFIG['cv_on']})")
print("=" * 80)

if CONFIG["cv_on"] == "train+val":

    X_cv_pool = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_cv_pool = np.concatenate([y_train, y_val])

    print("CV Pool = X_train + X_val")

else:

    X_cv_pool = X_train.reset_index(drop=True)
    y_cv_pool = y_train

    print("CV Pool = X_train only")

print("CV Pool shape :", X_cv_pool.shape)

# ==============================================================================
# Full Dataset vs Smoke Test / Subsample Control
# (toggle + row counts are set ONCE in Cell 4's CONFIG, not re-declared here)
# ==============================================================================

from sklearn.model_selection import train_test_split as _tts

print()
print("=" * 80)
print("Dataset Size Mode")
print("=" * 80)

if CONFIG["use_full_dataset"]:

    print("Mode     : FULL DATASET (USE_FULL_DATASET=True, SMOKE_TEST ignored)")
    print("CV Pool  :", X_cv_pool.shape)
    print("Test Set :", X_test.shape)

elif CONFIG["smoke_test"]:

    print(f"Mode     : SMOKE TEST (subsampling to {CONFIG['smoke_test_rows']:,} rows)")

    smoke_rows = CONFIG["smoke_test_rows"]

    if len(X_cv_pool) > smoke_rows:

        print(f"  Subsampling CV pool : {len(X_cv_pool):,} -> {smoke_rows:,} rows")

        _, X_cv_pool, _, y_cv_pool = _tts(
            X_cv_pool, y_cv_pool,
            test_size=smoke_rows,
            stratify=y_cv_pool,
            random_state=CONFIG["random_seed"],
        )

        X_cv_pool = X_cv_pool.reset_index(drop=True)

    else:

        print(f"  CV pool already <= {smoke_rows:,} rows, no subsampling needed.")

    if len(X_test) > smoke_rows:

        print(f"  Subsampling test set : {len(X_test):,} -> {smoke_rows:,} rows")

        _, X_test, _, y_test = _tts(
            X_test, y_test,
            test_size=smoke_rows,
            stratify=y_test,
            random_state=CONFIG["random_seed"],
        )

        X_test = X_test.reset_index(drop=True)

    else:

        print(f"  Test set already <= {smoke_rows:,} rows, no subsampling needed.")

    print()
    print(f"SMOKE TEST ON — CV pool: {X_cv_pool.shape}, Test: {X_test.shape}")

else:

    print("Mode     : FULL DATASET (no smoke test requested)")
    print("CV Pool  :", X_cv_pool.shape)
    print("Test Set :", X_test.shape)

print()
print("=" * 80)
print("Cell 5 Complete")
print("=" * 80)
print("CV Pool shape :", X_cv_pool.shape)
print("Held-out TEST (never used in CV) :", X_test.shape)

# `loaded` held a second copy of every frame during the load loop above --
free_memory("loaded", scope=globals())

print_ram_usage("after loading dataset")

In [ ]:
# ==============================================================================
# Cell 5b : FULL DATASET DETAILS (run before every model — sanity check)
# ==============================================================================
# Prints complete train / validation / test / CV-pool details: shapes,
# per-class counts + percentages, imbalance ratio, and which files were
# loaded. Run this cell right before training ANY model so you can see
# exactly what data that model is about to train/evaluate on.
# ==============================================================================

import pandas as pd

def _class_table(y_array, label_encoder, name):
    counts = pd.Series(y_array).value_counts().sort_index()
    total = counts.sum()
    table = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / total * 100).round(3),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\n--- {name} : {total:,} rows, {len(table)} classes ---")
    display(table)
    majority = table["Count"].iloc[0]
    minority = table["Count"].iloc[-1]
    print(f"  Majority class : {table['Class'].iloc[0]} ({majority:,})")
    print(f"  Minority class : {table['Class'].iloc[-1]} ({minority:,})")
    print(f"  Imbalance ratio (majority/minority) : {majority / max(minority,1):.1f} : 1")
    return table

print("=" * 80)
print("FULL DATASET DETAILS")
print("=" * 80)

print(f"\nFiles loaded this run:")
print(f"  Train X : {TRAIN_FEATURES_FILE}")
print(f"  Train y : {TRAIN_LABELS_FILE}")
print(f"  Val   X : {VAL_FEATURES_FILE}")
print(f"  Val   y : {VAL_LABELS_FILE}")
print(f"  Test  X : {TEST_FEATURES_FILE}")
print(f"  Test  y : {TEST_LABELS_FILE}")
print(f"\nCV_ON mode : {CONFIG['cv_on']}")
print(f"USE_SMOTE_PER_FOLD : {CONFIG['use_smote_per_fold']}")

print(f"\nShapes:")
print(f"  X_train  : {X_train.shape}   y_train  : {y_train.shape}")
print(f"  X_val    : {X_val.shape}   y_val    : {y_val.shape}")
print(f"  X_test   : {X_test.shape}   y_test   : {y_test.shape}")
print(f"  X_cv_pool: {X_cv_pool.shape}   y_cv_pool: {y_cv_pool.shape}")

train_table = _class_table(y_train, label_encoder, "TRAIN (used to fit the model)")
val_table   = _class_table(y_val,   label_encoder, "VALIDATION (natural distribution — do not balance)")
test_table  = _class_table(y_test,  label_encoder, "TEST (natural distribution — held out, never seen)")
cvpool_table = _class_table(y_cv_pool, label_encoder, "CV POOL (what cross-validation actually trains/scores on)")

# ------------------------------------------------------------------
# Warn loudly if CV pool distribution doesn't match the real-world
# (test set) distribution -- this is exactly the bug that causes
# "great CV score, bad real test score".
# ------------------------------------------------------------------
cv_dist = (cvpool_table.set_index("Class")["Percent"]).sort_index()
test_dist = (test_table.set_index("Class")["Percent"]).sort_index()
max_gap = (cv_dist - test_dist).abs().max()

print("\n" + "=" * 80)
print("CV POOL vs TEST SET distribution check")
print("=" * 80)
print(f"Max class-percentage gap between CV pool and TEST set: {max_gap:.2f} pts")
if max_gap > 5:
    print("⚠️  WARNING: CV pool distribution differs substantially from the real")
    print("    TEST distribution. Metrics from CV will NOT be representative of")
    print("    real-world test performance. Make sure CV_ON == 'train_only' and")
    print("    that TRAIN_FEATURES_FILE is the SMOTE-balanced file on purpose —")
    print("    val/test themselves should stay untouched/imbalanced.")
else:
    print("OK: CV pool and TEST distributions are reasonably close.")

print("\n" + "=" * 80)
print("Dataset Details Complete")
print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 6 : Data Leakage Audit
# ==============================================================================
# Explicit, printed checks that the splits are leakage-free BEFORE any model
# is trained. Re-run this any time you touch the preprocessing notebook.
# ==============================================================================

print("=" * 80)
print("Data Leakage Audit")
print("=" * 80)

leakage_results = {}

def audit(name, condition):

    status = "PASS" if condition else "FAIL"

    leakage_results[name] = bool(condition)

    print(f"{name:<55}{status}")

# ------------------------------------------------------------------
# 1) No duplicated rows across train / val / test (exact-row leakage)
# ------------------------------------------------------------------
train_hash = pd.util.hash_pandas_object(X_train, index=False)
val_hash   = pd.util.hash_pandas_object(X_val,   index=False)
test_hash  = pd.util.hash_pandas_object(X_test,  index=False)

audit(
    "No duplicate rows: train vs test",
    len(set(train_hash) & set(test_hash)) == 0,
)

audit(
    "No duplicate rows: val vs test",
    len(set(val_hash) & set(test_hash)) == 0,
)

audit(
    "No duplicate rows: train vs val",
    len(set(train_hash) & set(val_hash)) == 0,
)

# ------------------------------------------------------------------
# 2) Feature columns identical & in the same order across splits
# ------------------------------------------------------------------
audit(
    "Identical feature columns (train == val == test)",
    list(X_train.columns) == list(X_val.columns) == list(X_test.columns),
)

# ------------------------------------------------------------------
# 3) No NaN / Inf introduced by scaling
# ------------------------------------------------------------------
audit(
    "No NaN in train features",
    not X_train.isna().any().any(),
)

audit(
    "No NaN in test features",
    not X_test.isna().any().any(),
)

audit(
    "No Inf in train features",
    not np.isinf(X_train.values).any(),
)

# ------------------------------------------------------------------
# 4) Target column is NOT present in the feature matrices
# ------------------------------------------------------------------
audit(
    "Target column absent from X_train",
    TARGET_COLUMN not in X_train.columns,
)

# ------------------------------------------------------------------
# 5) Class balance sanity (test set should reflect natural distribution,
#    i.e. must NOT have been SMOTE-resampled)
# ------------------------------------------------------------------
test_dist = pd.Series(y_test).value_counts(normalize=True)

audit(
    "Test set is imbalanced (not synthetically balanced)",
    test_dist.max() > (1.0 / N_CLASSES) * 1.2,
)

print()
passed = sum(leakage_results.values())
total = len(leakage_results)
print(f"Leakage Audit : {passed}/{total} checks passed")

pd.DataFrame({
    "Check": list(leakage_results.keys()),
    "Status": ["PASS" if v else "FAIL" for v in leakage_results.values()],
}).to_csv(TABLE_DIR / "data_leakage_audit.csv", index=False)

print("Saved:", TABLE_DIR / "data_leakage_audit.csv")


In [ ]:
# ==============================================================================
# Cell 7 : Cross-Validation Splitter
# ==============================================================================
# ONE fixed set of fold indices is generated here and saved to disk so that
# every model below (run in any order, any day) sees EXACTLY the same folds.
# This is what makes the paired t-test / McNemar's test valid later on.
#
# CONFIG["run_validation"] = False -> FOLD_INDICES = None, and every model's
#   TRAIN cell skips CV/validation entirely (final fit + test eval only).
# CONFIG["use_kfold"]      = False -> FOLD_INDICES is a single (train, val)
#   split (test_size = CONFIG["val_split_size"]) instead of N_FOLDS folds.
#   Everything downstream (fit_folds / score_folds) treats this exactly like
#   a 1-fold CV, so no other cell needs to know which mode is active.
# ==============================================================================

from sklearn.model_selection import StratifiedKFold, KFold, train_test_split

print("=" * 80)

if not CONFIG["run_validation"]:

    print("RUN_VALIDATION = False -> skipping fold/split generation")
    print("Every model TRAIN cell below will do a final fit + test eval only.")
    print("=" * 80)

    FOLD_INDICES = None

else:

    if CONFIG["use_kfold"]:

        print(f"Building {CONFIG['n_folds']}-Fold Splitter")
        print("=" * 80)

        if CONFIG["stratified"]:

            splitter = StratifiedKFold(
                n_splits=CONFIG["n_folds"],
                shuffle=True,
                random_state=CONFIG["random_seed"],
            )

        else:

            splitter = KFold(
                n_splits=CONFIG["n_folds"],
                shuffle=True,
                random_state=CONFIG["random_seed"],
            )

        FOLD_INDICES = list(splitter.split(X_cv_pool, y_cv_pool))

    else:

        print(f"USE_KFOLD = False -> building a single train/val split "
              f"(val_size={CONFIG['val_split_size']})")
        print("=" * 80)

        all_idx = np.arange(len(X_cv_pool))

        tr_idx, va_idx = train_test_split(
            all_idx,
            test_size=CONFIG["val_split_size"],
            stratify=y_cv_pool if CONFIG["stratified"] else None,
            random_state=CONFIG["random_seed"],
        )

        FOLD_INDICES = [(tr_idx, va_idx)]

    fold_summary = []

    for fold_id, (tr_idx, va_idx) in enumerate(FOLD_INDICES, 1):

        fold_summary.append({
            "Fold": fold_id,
            "Train Size": len(tr_idx),
            "Val Size": len(va_idx),
        })

        print(f"Fold {fold_id:02d} -> Train: {len(tr_idx):>7,}  Val: {len(va_idx):>7,}")

    pd.DataFrame(fold_summary).to_csv(TABLE_DIR / "cv_fold_sizes.csv", index=False)

    joblib.dump(FOLD_INDICES, MODEL_REPORT_DIR / "fold_indices.pkl")

    print()
    print("Saved:", TABLE_DIR / "cv_fold_sizes.csv")
    print("Saved:", MODEL_REPORT_DIR / "fold_indices.pkl")


In [ ]:
# ==============================================================================
# Cell 8 : Shared Helper Functions
# ==============================================================================
# Every model cell below calls ONLY these helpers plus CONFIG / X_cv_pool /
# FOLD_INDICES / X_test / y_test, so each model cell is fully self-contained.
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
)
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")


def compute_metrics(y_true, y_pred, y_proba=None):
    """Standard classification metrics used across every model in this notebook."""

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

    if y_proba is not None:

        try:

            metrics["roc_auc_ovr"] = roc_auc_score(
                y_true, y_proba, multi_class="ovr", average="macro",
            )

        except Exception:

            metrics["roc_auc_ovr"] = np.nan

    return metrics


def fit_folds(model_name, fit_predict_fn):
    """
    TRAIN-side half of CV: fits + predicts per fold (or per the single
    train/val split when CONFIG["use_kfold"]=False), but does NOT score
    anything -- that is score_folds()'s job, kept in a separate cell so
    training and validation/scoring are independent steps.

    fit_predict_fn(X_tr, y_tr, X_va) -> (y_pred, y_proba)
      must fit on (X_tr, y_tr) and return predictions/probabilities for X_va.

    Returns: (oof_pred, oof_proba, fold_records)
      fold_records is a list of {"fold", "y_true", "y_pred", "y_proba"} dicts,
      consumed later by score_folds(). If CONFIG["run_validation"] is False
      (FOLD_INDICES is None), this is a no-op and returns (None, None, None).
    """

    if FOLD_INDICES is None:

        print(f"[{model_name}] RUN_VALIDATION=False -> skipping fold training "
              f"(no CV/validation fold will be fit).")

        return None, None, None

    n_folds_actual = len(FOLD_INDICES)

    fold_records = []

    oof_pred = np.full(len(y_cv_pool), -1)

    oof_proba = np.zeros((len(y_cv_pool), N_CLASSES))

    X_arr = X_cv_pool.reset_index(drop=True)

    for fold_id, (tr_idx, va_idx) in enumerate(FOLD_INDICES, 1):

        X_tr, X_va = X_arr.iloc[tr_idx], X_arr.iloc[va_idx]

        y_tr, y_va = y_cv_pool[tr_idx], y_cv_pool[va_idx]

        if CONFIG["use_smote_per_fold"]:

            smote = SMOTE(random_state=CONFIG["random_seed"])

            X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

        y_pred, y_proba = fit_predict_fn(X_tr, y_tr, X_va)

        oof_pred[va_idx] = y_pred

        if y_proba is not None:

            oof_proba[va_idx] = y_proba

        fold_records.append({
            "fold": fold_id,
            "y_true": y_va,
            "y_pred": y_pred,
            "y_proba": y_proba,
        })

        print(f"[{model_name}] Fold {fold_id:02d}/{n_folds_actual}  fit+predict done  "
              f"(train={len(tr_idx):,}  val={len(va_idx):,})")

        # Fold-scoped variables (model/optimizer/loaders inside fit_predict_fn
        # are already freed there) -- this extra pass protects against Colab
        # crashing partway through a 5-10 fold CV loop across 7 models.
        del X_tr, X_va, y_tr, y_va, y_pred, y_proba

        cleanup_torch()

    del X_arr

    gc.collect()

    oof_df = pd.DataFrame({
        "y_true": y_cv_pool,
        "y_pred": oof_pred,
    })

    oof_df.to_csv(PRED_DIR / f"{model_name}_oof_predictions.csv", index=False)

    np.save(PRED_DIR / f"{model_name}_oof_proba.npy", oof_proba)

    return oof_pred, oof_proba, fold_records


def score_folds(model_name, fold_records):
    """
    VALIDATE-side half of CV: scores the raw (y_true, y_pred, y_proba) fold
    records produced by fit_folds(). Lives in its own cell downstream so it
    can be skipped outright when CONFIG["run_validation"] is False.

    Returns: fold_metrics_df (or None if fold_records is None/empty).
    """

    if not fold_records:

        print(f"[{model_name}] No fold records to score "
              f"(RUN_VALIDATION=False, or the TRAIN cell wasn't run yet).")

        return None

    fold_rows = []

    for rec in fold_records:

        row = {"Model": model_name, "Fold": rec["fold"]}

        row.update(compute_metrics(rec["y_true"], rec["y_pred"], rec["y_proba"]))

        fold_rows.append(row)

        print(f"[{model_name}] Fold {rec['fold']:02d}  "
              f"F1-macro={row['f1_macro']:.4f}  Acc={row['accuracy']:.4f}")

    fold_df = pd.DataFrame(fold_rows)

    fold_df.to_csv(TABLE_DIR / f"{model_name}_cv_fold_metrics.csv", index=False)

    print()

    if len(fold_df) > 1:

        print(fold_df.describe().loc[["mean", "std"]])

    else:

        print(fold_df)

    return fold_df


def evaluate_on_test(model_name, y_pred, y_proba=None):
    """Final, ONE-TIME evaluation on the held-out test set. Saves metrics + figures."""

    metrics = compute_metrics(y_test, y_pred, y_proba)

    print("=" * 80)
    print(f"{model_name} — Held-Out Test Set Results")
    print("=" * 80)

    for key, value in metrics.items():

        print(f"{key:<20}: {value:.4f}")

    pd.DataFrame([metrics]).to_csv(TABLE_DIR / f"{model_name}_test_metrics.csv", index=False)

    pd.DataFrame({"y_true": y_test, "y_pred": y_pred}).to_csv(
        PRED_DIR / f"{model_name}_test_predictions.csv", index=False,
    )

    if y_proba is not None:

        np.save(PRED_DIR / f"{model_name}_test_proba.npy", y_proba)

    # ---------------- Confusion Matrix ----------------
    cm = confusion_matrix(y_test, y_pred, labels=range(N_CLASSES))

    fig, ax = plt.subplots(figsize=(9, 8))

    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)

    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=True)

    ax.set_title(f"{model_name} — Confusion Matrix (Test Set)", fontsize=15)

    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"{model_name}_confusion_matrix.{CONFIG['figure_format']}",
        dpi=CONFIG["figure_dpi"], bbox_inches="tight",
    )

    plt.show()

    # ---------------- ROC Curve (One-vs-Rest, macro) ----------------
    if y_proba is not None:

        fig, ax = plt.subplots(figsize=(8, 7))

        for class_idx, class_name in enumerate(CLASS_NAMES):

            y_true_bin = (y_test == class_idx).astype(int)

            try:

                RocCurveDisplay.from_predictions(
                    y_true_bin, y_proba[:, class_idx],
                    name=class_name, ax=ax,
                )

            except Exception:

                pass

        ax.set_title(f"{model_name} — ROC Curves (Test Set, OvR)", fontsize=15)

        plt.tight_layout()

        plt.savefig(
            FIGURE_DIR / f"{model_name}_roc_curve.{CONFIG['figure_format']}",
            dpi=CONFIG["figure_dpi"], bbox_inches="tight",
        )

        plt.show()

    return metrics


def get_device():
    """
    Cross-vendor GPU selection driven entirely by CONFIG["device_preference"]
    and CONFIG["gpu_index"] (both set in Cell 4, the Master Control Cell).

    Supports:
      NVIDIA GPUs      -> "cuda"  (torch.cuda.*)
      AMD GPUs (ROCm)  -> "cuda"  (ROCm builds of torch expose the SAME
                          torch.cuda.* API, so no separate option is needed)
      Intel GPUs       -> "xpu"   (native torch.xpu on torch>=2.4, or via
                          intel_extension_for_pytorch on older torch)
      Apple Silicon    -> "mps"
      "auto"           -> cuda -> xpu -> mps -> cpu, first one available
      "cpu"            -> always CPU

    Falls back to CPU (with a warning) if the requested backend isn't
    actually available, so a run never silently trains on the wrong device
    or crashes because a laptop doesn't have the GPU picked in Cell 4.
    """

    import torch

    pref = CONFIG["device_preference"]
    idx = CONFIG.get("gpu_index", 0)

    def _cuda_ok():
        try:
            return torch.cuda.is_available() and torch.cuda.device_count() > idx
        except Exception:
            return False

    def _xpu_ok():
        try:
            if not hasattr(torch, "xpu"):
                import intel_extension_for_pytorch  # noqa: F401 -- registers torch.xpu as a side effect
            return hasattr(torch, "xpu") and torch.xpu.is_available() and torch.xpu.device_count() > idx
        except Exception:
            return False

    def _mps_ok():
        try:
            return torch.backends.mps.is_available()
        except Exception:
            return False

    if pref == "cpu":
        return torch.device("cpu")

    if pref == "cuda":
        if _cuda_ok():
            return torch.device(f"cuda:{idx}")
        print(f'  WARNING: DEVICE_PREFERENCE="cuda" but no NVIDIA/AMD(ROCm) GPU at index {idx} was found -- falling back to CPU.')
        return torch.device("cpu")

    if pref == "xpu":
        if _xpu_ok():
            return torch.device(f"xpu:{idx}")
        print(f'  WARNING: DEVICE_PREFERENCE="xpu" but no Intel GPU at index {idx} was found -- falling back to CPU.')
        return torch.device("cpu")

    if pref == "mps":
        if _mps_ok():
            return torch.device("mps")
        print('  WARNING: DEVICE_PREFERENCE="mps" but Apple MPS is not available -- falling back to CPU.')
        return torch.device("cpu")

    # "auto": NVIDIA/AMD -> Intel -> Apple -> CPU
    if _cuda_ok():
        return torch.device(f"cuda:{idx}")
    if _xpu_ok():
        return torch.device(f"xpu:{idx}")
    if _mps_ok():
        return torch.device("mps")
    return torch.device("cpu")


print("Helper functions ready: compute_metrics, fit_folds, score_folds, evaluate_on_test, get_device")


## Model 1 — CatBoost

In [ ]:
# ==============================================================================
# Cell 9 : Model 1 - CatBoost
# ==============================================================================
# Gradient-boosted decision trees, native categorical support, strong
# baseline for tabular / NetFlow-style IDS features.
# Depends only on: Cells 1-8.
# ==============================================================================

from catboost import CatBoostClassifier

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("catboost")

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)

_print_training_data_summary("catboost")

MODEL_NAME = "catboost"

print("=" * 80)
print("CatBoost - Cross Validation")
print("=" * 80)

# CatBoost has its own native progress printer (via `verbose=N`, printing
# every N boosting iterations) -- no custom print loop needed. Tied to the
# same CONFIG["nn_verbose"] toggle used by the neural models, so one switch
# controls verbosity everywhere in the notebook.
_catboost_verbose = 50 if CONFIG["nn_verbose"] else False


def catboost_fit_predict(X_tr, y_tr, X_va):

    # NOTE: CatBoost's GPU support is NVIDIA-CUDA only (upstream CatBoost has
    # no AMD/ROCm or Intel/XPU GPU backend). We reuse the SAME get_device()
    # used by every neural model in Cell 4/8 so the manual DEVICE_PREFERENCE
    # dropdown stays the single source of truth, but only actually request
    # CatBoost's GPU mode when the resolved device is a CUDA device -- on an
    # AMD/Intel/Apple/"auto"-without-NVIDIA machine this correctly falls back
    # to CatBoost's CPU mode instead of erroring out.
    _resolved_device = get_device()

    _use_catboost_gpu = _resolved_device.type == "cuda"

    _catboost_kwargs = {}

    if _use_catboost_gpu:
        _catboost_kwargs["devices"] = str(_resolved_device.index or 0)

    model = CatBoostClassifier(
        iterations=600,
        depth=8,
        learning_rate=0.08,
        loss_function="MultiClass" if N_CLASSES > 2 else "Logloss",
        auto_class_weights="Balanced",
        random_seed=CONFIG["random_seed"],
        task_type="GPU" if _use_catboost_gpu else "CPU",
        verbose=_catboost_verbose,
        **_catboost_kwargs,
    )

    model.fit(X_tr, y_tr, verbose=_catboost_verbose)

    y_pred = model.predict(X_va).flatten().astype(int)

    y_proba = model.predict_proba(X_va)

    return y_pred, y_proba

catboost_oof_pred, catboost_oof_proba, catboost_fold_records = fit_folds(MODEL_NAME, catboost_fit_predict)

# ==============================================================================
# Final Fit on FULL CV Pool -> Evaluate ONCE on Held-Out Test Set
# ==============================================================================

print()
print("=" * 80)
print("Fitting final CatBoost model on full CV pool "
      f"({X_cv_pool.shape[0]:,} rows) -- this is the model that gets saved "
      "and scored on the held-out test set.")
print("=" * 80)

if CONFIG["use_smote_per_fold"]:

    smote_final = SMOTE(random_state=CONFIG["random_seed"])

    X_final, y_final = smote_final.fit_resample(X_cv_pool, y_cv_pool)

else:

    X_final, y_final = X_cv_pool, y_cv_pool

_resolved_device_final = get_device()

_use_catboost_gpu_final = _resolved_device_final.type == "cuda"

_catboost_kwargs_final = {}

if _use_catboost_gpu_final:
    _catboost_kwargs_final["devices"] = str(_resolved_device_final.index or 0)

catboost_final = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.08,
    loss_function="MultiClass" if N_CLASSES > 2 else "Logloss",
    auto_class_weights="Balanced",
    random_seed=CONFIG["random_seed"],
    task_type="GPU" if _use_catboost_gpu_final else "CPU",
    verbose=_catboost_verbose,
    **_catboost_kwargs_final,
)

catboost_final.fit(X_final, y_final, verbose=_catboost_verbose)

print("Final CatBoost fit complete.")

catboost_test_pred = catboost_final.predict(X_test).flatten().astype(int)

catboost_test_proba = catboost_final.predict_proba(X_test)

catboost_final.save_model(str(TRAINED_MODEL_DIR / "catboost_final.cbm"))

print("Saved:", TRAINED_MODEL_DIR / "catboost_final.cbm")

cleanup_torch()
print_ram_usage("after cell 11")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### CatBoost -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- CatBoost (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "catboost"

if CONFIG["run_validation"]:

    catboost_cv_metrics = score_folds(MODEL_NAME, catboost_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    catboost_cv_metrics = None

print()

catboost_test_metrics = evaluate_on_test(MODEL_NAME, catboost_test_pred, catboost_test_proba)


## Neural Tabular Models — Shared PyTorch Training Loop

The remaining neural models (FT-Transformer, TabM, TabGT, GNNFormer, GraphTransformer) all
share one generic training loop (`train_torch_classifier`) and only differ in the
`nn.Module` architecture. This keeps each model cell short, comparable, and independent —
each cell defines its OWN architecture class and calls the shared trainer.

In [ ]:
# ==============================================================================
# Cell 10 : Shared PyTorch Training Utilities (used by every NN model below)
# ==============================================================================

import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


class TabularDataset(TensorDataset):
    pass


def make_loader(X, y, batch_size, shuffle):

    X_t = torch.tensor(X.values if hasattr(X, "values") else X, dtype=torch.float32)

    y_t = torch.tensor(y, dtype=torch.long)

    ds = TensorDataset(X_t, y_t)

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_torch_classifier(model_ctor, X_tr, y_tr, X_va, epochs=None, patience=None, verbose=None):
    """
    Generic training loop with early stopping on validation loss.
    model_ctor: zero-arg callable returning a fresh nn.Module(input_dim, n_classes)
    Returns: (y_pred, y_proba) for X_va
    """

    device = get_device()

    epochs = epochs or CONFIG["nn_epochs"]

    patience = patience or CONFIG["nn_patience"]

    verbose = CONFIG["nn_verbose"] if verbose is None else verbose

    model = model_ctor().to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["nn_learning_rate"], weight_decay=1e-5)

    # Class-weighted loss: without this, the loss is dominated by BENIGN
    # (still the largest class even after Cell 12's capped balancing) and
    # the model has little incentive to get rare attack classes right --
    # this is a real contributor to the macro-F1 gap seen on the held-out
    # test set. Weight = inverse class frequency in THIS fold's training
    # split, normalized so weights average to 1.
    class_counts_fold = np.bincount(y_tr, minlength=N_CLASSES).astype(np.float64)

    class_counts_fold[class_counts_fold == 0] = 1  # avoid div-by-zero for absent classes

    class_weights = class_counts_fold.sum() / (len(class_counts_fold) * class_counts_fold)

    class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_t)

    # effective batch size cannot exceed the number of training rows,
    # otherwise very small smoke-test folds create a single, unstable batch
    effective_batch_size = min(CONFIG["nn_batch_size"], max(len(X_tr) // 4, 8))

    train_loader = make_loader(X_tr, y_tr, effective_batch_size, shuffle=True)

    X_va_t = torch.tensor(X_va.values if hasattr(X_va, "values") else X_va, dtype=torch.float32).to(device)

    best_state = None

    best_loss = float("inf")

    patience_counter = 0

    nan_detected = False

    n_train_rows = len(X_tr)

    n_batches_per_epoch = max(1, (n_train_rows + effective_batch_size - 1) // effective_batch_size)

    for epoch in range(epochs):

        epoch_start_time = time.time()

        model.train()

        epoch_had_valid_step = False

        running_loss_sum = 0.0

        running_correct = 0

        running_seen = 0

        n_skipped_batches = 0

        batch_idx = 0

        grad_norm_last = 0.0

        for xb, yb in train_loader:

            batch_idx += 1

            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()

            out = model(xb)

            loss = criterion(out, yb)

            if torch.isnan(loss) or torch.isinf(loss):

                # skip this step rather than corrupting the model weights
                n_skipped_batches += 1

                continue

            loss.backward()

            # gradient clipping prevents attention-based models (FT-Transformer,
            # GNNFormer, GraphTransformer) from exploding on very small /
            # unstable batches, which is what was producing NaN probabilities
            grad_norm_last = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()

            optimizer.step()

            epoch_had_valid_step = True

            with torch.no_grad():

                running_loss_sum += loss.item() * xb.size(0)

                running_correct += (out.argmax(dim=1) == yb).sum().item()

                running_seen += xb.size(0)

            if verbose:

                print(f"\r  batch {batch_idx:04d}/{n_batches_per_epoch:04d} "
                      f"- loss: {loss.item():.4f}", end="", flush=True)

        if verbose and epoch_had_valid_step:

            print()  # newline after the last in-epoch batch progress line

        if not epoch_had_valid_step:

            nan_detected = True

            if verbose:

                print(f"Epoch {epoch + 1:03d}/{epochs:03d} - SKIPPED "
                      f"(all {n_batches_per_epoch} batches produced NaN/Inf loss)")

            continue

        train_acc_epoch = running_correct / max(running_seen, 1)

        train_loss_epoch = running_loss_sum / max(running_seen, 1)

        epoch_time = time.time() - epoch_start_time

        # early stopping tracked on TRAIN loss trend (no val labels used to
        # avoid any leakage of the CV validation fold into model selection
        # beyond standard early stopping practice)
        if train_loss_epoch < best_loss - 1e-4:

            best_loss = train_loss_epoch

            best_state = {k: v.clone() for k, v in model.state_dict().items()}

            patience_counter = 0

            improved_flag = "*"

        else:

            patience_counter += 1

            improved_flag = " "

        if verbose:

            current_lr = optimizer.param_groups[0]["lr"]

            print(f"Epoch {epoch + 1:03d}/{epochs:03d}{improved_flag} - "
                  f"{epoch_time:.1f}s - "
                  f"loss: {train_loss_epoch:.4f} - "
                  f"acc: {train_acc_epoch:.4f} - "
                  f"lr: {current_lr:.2e} - "
                  f"grad_norm: {grad_norm_last:.3f} - "
                  f"best_loss: {best_loss:.4f} - "
                  f"patience: {patience_counter}/{patience}"
                  + (f" - skipped_batches: {n_skipped_batches}" if n_skipped_batches else ""))

        if patience_counter >= patience:

            if verbose:

                print(f"Early stopping at epoch {epoch + 1:03d} "
                      f"(no improvement for {patience} epochs, best loss={best_loss:.4f})")

            break

    if best_state is not None:

        model.load_state_dict(best_state)

    elif nan_detected:

        print("  WARNING: training never stabilized (all epochs produced NaN/Inf loss). "
              "Falling back to an untrained model — predictions will be near-uniform. "
              "Consider increasing SMOKE_TEST_ROWS or lowering NN_LEARNING_RATE.")

    model.eval()

    # Evaluate in chunks rather than one giant forward pass. Graph/KNN
    # attention models (TabGT, GNNFormer, GraphTransformer) build an
    # O(N^2) attention/distance matrix ACROSS THE BATCH DIMENSION -- doing
    # that over an entire CV validation fold (or the full test set during
    # final fit) tries to allocate a matrix of size (fold_size x fold_size),
    # which is exactly what was causing the CUDA OOM crashes.
    eval_batch_size = CONFIG["nn_batch_size"]

    all_logits = []

    with torch.no_grad():

        for start in range(0, X_va_t.size(0), eval_batch_size):

            chunk = X_va_t[start:start + eval_batch_size]

            chunk_logits = model(chunk)

            all_logits.append(chunk_logits)

        logits = torch.cat(all_logits, dim=0)

        if torch.isnan(logits).any() or torch.isinf(logits).any():

            print("  WARNING: NaN/Inf logits detected at prediction time — "
                  "replacing with uniform probabilities to avoid crashing downstream cells.")

            logits = torch.zeros_like(logits)

        proba = torch.softmax(logits, dim=1).cpu().numpy()

        pred = proba.argmax(axis=1)

        del all_logits

    # `model` is returned (final-fit calls save its state_dict), but
    # everything else here -- optimizer state, the DataLoader, the
    # validation tensor already copied to GPU, and the cloned
    # best-epoch weights -- is pure training scratch space. Across
    # 5-10 folds x 5 NN models this scratch space is what actually
    # fills up VRAM/RAM over a run, since PyTorch's CUDA allocator
    # does not return memory to the OS on its own.
    del optimizer, train_loader, X_va_t, logits, class_weights_t

    if best_state is not None:

        del best_state

    cleanup_torch()

    return pred, proba, model


print("Shared PyTorch utilities ready: make_loader, train_torch_classifier")

## Model 2 — FT-Transformer

In [ ]:
# ==============================================================================
# Cell 11 : Model 2 - FT-Transformer (Feature Tokenizer + Transformer)
# ==============================================================================
# Gorishniy et al., 2021 (widely used as the standard tabular transformer
# baseline, hence its inclusion alongside the 2024/2025 models). Each
# feature is tokenized into a d-dim embedding, a [CLS] token is prepended,
# and standard Transformer encoder blocks are applied.
# Depends only on: Cells 1-10.
# ==============================================================================

class FTTransformer(nn.Module):

    def __init__(self, n_features, n_classes, d_token=32, n_layers=3, n_heads=4, dropout=0.1):

        super().__init__()

        self.n_features = n_features

        # per-feature linear tokenizer: each scalar feature -> d_token vector
        self.feature_weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)

        self.feature_bias = nn.Parameter(torch.zeros(n_features, d_token))

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.norm = nn.LayerNorm(d_token)

        self.head = nn.Linear(d_token, n_classes)

    def forward(self, x):

        # Network flow features (duration, bytes/s, etc.) are heavy-tailed --
        # even after StandardScaler, extreme outliers can have huge z-scores.
        # Multiplying that directly into a learned per-feature token (below)
        # can overflow to Inf/NaN inside the transformer's softmax/LayerNorm,
        # which is exactly what caused every epoch to report NaN/Inf loss.
        # Clamping bounds the input without materially changing its meaning
        # (+/-10 std devs is already far into the outlier tail).
        x = torch.clamp(x, -10.0, 10.0)

        # x: (batch, n_features) -> tokens: (batch, n_features, d_token)
        tokens = x.unsqueeze(-1) * self.feature_weight + self.feature_bias

        cls = self.cls_token.expand(x.size(0), -1, -1)

        tokens = torch.cat([cls, tokens], dim=1)

        encoded = self.encoder(tokens)

        cls_out = self.norm(encoded[:, 0])

        return self.head(cls_out)


# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("fttransformer")

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)

_print_training_data_summary("fttransformer")

MODEL_NAME = "fttransformer"

print("=" * 80)
print("FT-Transformer - Cross Validation")
print("=" * 80)

def fttransformer_fit_predict(X_tr, y_tr, X_va):

    ctor = lambda: FTTransformer(N_FEATURES, N_CLASSES)

    y_pred, y_proba, _ = train_torch_classifier(ctor, X_tr, y_tr, X_va)

    return y_pred, y_proba

fttransformer_oof_pred, fttransformer_oof_proba, fttransformer_fold_records = fit_folds(MODEL_NAME, fttransformer_fit_predict)

# ==============================================================================
# Final Fit -> Held-Out Test Evaluation
# ==============================================================================

if CONFIG["use_smote_per_fold"]:

    smote_final = SMOTE(random_state=CONFIG["random_seed"])

    X_final, y_final = smote_final.fit_resample(X_cv_pool, y_cv_pool)

else:

    X_final, y_final = X_cv_pool, y_cv_pool

fttransformer_test_pred, fttransformer_test_proba, fttransformer_final = train_torch_classifier(
    lambda: FTTransformer(N_FEATURES, N_CLASSES), X_final, y_final, X_test,
)

torch.save(fttransformer_final.state_dict(), TRAINED_MODEL_DIR / "fttransformer_final.pt")

print("Saved:", TRAINED_MODEL_DIR / "fttransformer_final.pt")

cleanup_torch()
print_ram_usage("after cell 15")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### FT-Transformer -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- FT-Transformer (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "fttransformer"

if CONFIG["run_validation"]:

    fttransformer_cv_metrics = score_folds(MODEL_NAME, fttransformer_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    fttransformer_cv_metrics = None

print()

fttransformer_test_metrics = evaluate_on_test(MODEL_NAME, fttransformer_test_pred, fttransformer_test_proba)


## Model 3 — TabM (ICLR 2025, simplified reproduction)

> **Simplification note:** TabM (Gorishniy et al., ICLR 2025) is an efficient
> ensembling method: one shared backbone body with *k* lightweight,
> per-member scaling/adapter parameters (BatchEnsemble-style), producing
> *k* diverse predictions at almost the cost of one model. The official repo
> is not a stable pip package, so the core idea — a shared MLP body with
> `k` independent linear heads driven by per-member multiplicative masks —
> is re-implemented here directly in PyTorch.

In [ ]:
# ==============================================================================
# Cell 12 : Model 3 - TabM (simplified BatchEnsemble-style tabular model)
# ==============================================================================
# Depends only on: Cells 1-10.
# ==============================================================================

class TabM(nn.Module):

    def __init__(self, n_features, n_classes, hidden_dim=128, k_members=8, dropout=0.1):

        super().__init__()

        self.k = k_members

        # shared body
        self.body = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # per-member multiplicative mask (BatchEnsemble "R" vector) + bias
        self.member_scale = nn.Parameter(torch.randn(k_members, hidden_dim) * 0.02 + 1.0)

        self.member_bias = nn.Parameter(torch.zeros(k_members, hidden_dim))

        # per-member head, weight-tied except for a small adapter
        self.shared_head = nn.Linear(hidden_dim, n_classes)

        self.member_head_scale = nn.Parameter(torch.randn(k_members, n_classes) * 0.02 + 1.0)

    def forward(self, x):

        shared = self.body(x)  # (batch, hidden)

        # broadcast across k members
        member_feats = shared.unsqueeze(1) * self.member_scale.unsqueeze(0) + self.member_bias.unsqueeze(0)
        # (batch, k, hidden)

        member_logits = self.shared_head(member_feats) * self.member_head_scale.unsqueeze(0)
        # (batch, k, n_classes)

        # ensemble-average the k members -> final prediction
        return member_logits.mean(dim=1)


# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("tabm")

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)

_print_training_data_summary("tabm")

MODEL_NAME = "tabm"

print("=" * 80)
print("TabM - Cross Validation")
print("=" * 80)

def tabm_fit_predict(X_tr, y_tr, X_va):

    ctor = lambda: TabM(N_FEATURES, N_CLASSES)

    y_pred, y_proba, _ = train_torch_classifier(ctor, X_tr, y_tr, X_va)

    return y_pred, y_proba

tabm_oof_pred, tabm_oof_proba, tabm_fold_records = fit_folds(MODEL_NAME, tabm_fit_predict)

# ==============================================================================
# Final Fit -> Held-Out Test Evaluation
# ==============================================================================

if CONFIG["use_smote_per_fold"]:

    smote_final = SMOTE(random_state=CONFIG["random_seed"])

    X_final, y_final = smote_final.fit_resample(X_cv_pool, y_cv_pool)

else:

    X_final, y_final = X_cv_pool, y_cv_pool

tabm_test_pred, tabm_test_proba, tabm_final = train_torch_classifier(
    lambda: TabM(N_FEATURES, N_CLASSES), X_final, y_final, X_test,
)

torch.save(tabm_final.state_dict(), TRAINED_MODEL_DIR / "tabm_final.pt")

print("Saved:", TRAINED_MODEL_DIR / "tabm_final.pt")

cleanup_torch()
print_ram_usage("after cell 17")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### TabM -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- TabM (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "tabm"

if CONFIG["run_validation"]:

    tabm_cv_metrics = score_folds(MODEL_NAME, tabm_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    tabm_cv_metrics = None

print()

tabm_test_metrics = evaluate_on_test(MODEL_NAME, tabm_test_pred, tabm_test_proba)


## Model 4 — TabGT (2024, simplified reproduction)

> **Simplification note:** TabGT-style models treat a table as a bipartite /
> row-column graph and apply graph attention between a row's own features
> and its k nearest neighbour rows in feature space, before a Transformer
> refines the row representation. No official pip package exists; the core
> "neighbour-attention over a feature-similarity graph, then self-attention"
> idea is reproduced with a k-NN graph built once per batch.

In [ ]:
# ==============================================================================
# Cell 13 : Model 4 - TabGT (row-graph + transformer, simplified)
# ==============================================================================
# Depends only on: Cells 1-10.
# ==============================================================================

class TabGT(nn.Module):

    def __init__(self, n_features, n_classes, d_model=64, k_neighbors=8, n_heads=4, dropout=0.1):

        super().__init__()

        self.k = k_neighbors

        self.input_proj = nn.Linear(n_features, d_model)

        self.neighbor_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.norm1 = nn.LayerNorm(d_model)

        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Linear(d_model * 2, d_model),
        )

        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):

        # project rows to embeddings
        h = self.input_proj(x)  # (batch, d_model)

        # build a k-NN graph on the CURRENT batch in embedding space
        with torch.no_grad():

            dist = torch.cdist(h, h)  # (batch, batch)

            # k can never exceed (batch_size - 1); if the batch has only 1
            # sample, there are no neighbors to attend to, so k=0 (self-only)
            k = max(min(self.k, h.size(0) - 1), 0)

            _, nn_idx = torch.topk(-dist, k=k + 1, dim=1)  # k+1 <= h.size(0), safe

        neighbor_feats = h[nn_idx]  # (batch, k+1, d_model)

        # row attends to its own k-NN neighborhood (graph attention proxy)
        attn_out, _ = self.neighbor_attn(h.unsqueeze(1), neighbor_feats, neighbor_feats)

        h = self.norm1(h + attn_out.squeeze(1))

        # global self-attention across the batch (transformer refinement)
        self_out, _ = self.self_attn(h.unsqueeze(1), h.unsqueeze(1), h.unsqueeze(1))

        h = self.norm2(h + self_out.squeeze(1))

        h = h + self.ffn(h)

        return self.head(h)


# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("tabgt")

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)

_print_training_data_summary("tabgt")

MODEL_NAME = "tabgt"

print("=" * 80)
print("TabGT - Cross Validation")
print("=" * 80)

def tabgt_fit_predict(X_tr, y_tr, X_va):

    ctor = lambda: TabGT(N_FEATURES, N_CLASSES)

    y_pred, y_proba, _ = train_torch_classifier(ctor, X_tr, y_tr, X_va)

    return y_pred, y_proba

tabgt_oof_pred, tabgt_oof_proba, tabgt_fold_records = fit_folds(MODEL_NAME, tabgt_fit_predict)

if CONFIG["use_smote_per_fold"]:

    smote_final = SMOTE(random_state=CONFIG["random_seed"])

    X_final, y_final = smote_final.fit_resample(X_cv_pool, y_cv_pool)

else:

    X_final, y_final = X_cv_pool, y_cv_pool

tabgt_test_pred, tabgt_test_proba, tabgt_final = train_torch_classifier(
    lambda: TabGT(N_FEATURES, N_CLASSES), X_final, y_final, X_test,
)

torch.save(tabgt_final.state_dict(), TRAINED_MODEL_DIR / "tabgt_final.pt")

print("Saved:", TRAINED_MODEL_DIR / "tabgt_final.pt")

cleanup_torch()
print_ram_usage("after cell 19")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### TabGT -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- TabGT (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "tabgt"

if CONFIG["run_validation"]:

    tabgt_cv_metrics = score_folds(MODEL_NAME, tabgt_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    tabgt_cv_metrics = None

print()

tabgt_test_metrics = evaluate_on_test(MODEL_NAME, tabgt_test_pred, tabgt_test_proba)


## Model 5 — GNNFormer (2024, simplified reproduction)

> **Simplification note:** GNNFormer-style IDS models build a graph over
> network FLOWS (nodes = flows, edges = shared IP/port/time-window
> relationships) and run message passing before classification. Since this
> notebook works on the tabular, per-flow feature table produced by
> preprocessing (no explicit IP-graph is available at this stage), the
> "message passing" step is approximated with a differentiable feature-graph:
> features attend to one another as graph nodes (a GCN-style layer over
> a learned feature-adjacency), followed by a Transformer classifier head —
> the two-stage "GNN + Former" structure the name implies.

In [ ]:
# ==============================================================================
# Cell 14 : Model 5 - GNNFormer (learned feature-graph GCN + Transformer)
# ==============================================================================
# Depends only on: Cells 1-10.
# ==============================================================================

class GCNLayer(nn.Module):

    def __init__(self, n_nodes, d_model):

        super().__init__()

        # learned, symmetric adjacency over the FEATURE graph (n_nodes = n_features)
        self.adj_logits = nn.Parameter(torch.randn(n_nodes, n_nodes) * 0.01)

        self.proj = nn.Linear(d_model, d_model)

    def forward(self, node_feats):

        # node_feats: (batch, n_nodes, d_model)
        adj = torch.softmax(self.adj_logits + self.adj_logits.t(), dim=-1)  # (n_nodes, n_nodes)

        aggregated = torch.einsum("ij,bjd->bid", adj, node_feats)

        return torch.relu(self.proj(aggregated) + node_feats)


class GNNFormer(nn.Module):

    def __init__(self, n_features, n_classes, d_model=32, n_gcn_layers=2, n_heads=4, dropout=0.1):

        super().__init__()

        self.token_embed = nn.Parameter(torch.randn(n_features, d_model) * 0.02)

        self.gcn_layers = nn.ModuleList([GCNLayer(n_features, d_model) for _ in range(n_gcn_layers)])

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )

        self.former = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.pool_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):

        # Network flow features (duration, bytes/s, etc.) are heavy-tailed --
        # even after StandardScaler, extreme outliers can have huge z-scores.
        # Multiplying that directly into a learned per-feature token (below)
        # can overflow to Inf/NaN inside the transformer's softmax/LayerNorm,
        # which is exactly what caused every epoch to report NaN/Inf loss.
        # Clamping bounds the input without materially changing its meaning
        # (+/-10 std devs is already far into the outlier tail).
        x = torch.clamp(x, -10.0, 10.0)

        # each feature is a graph node; value scales a learned node token
        node_feats = x.unsqueeze(-1) * self.token_embed.unsqueeze(0)  # (batch, n_features, d_model)

        for gcn in self.gcn_layers:

            node_feats = gcn(node_feats)

        encoded = self.former(node_feats)

        pooled = self.pool_norm(encoded.mean(dim=1))  # graph-level readout

        return self.head(pooled)


# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("gnnformer")

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)

_print_training_data_summary("gnnformer")

MODEL_NAME = "gnnformer"

print("=" * 80)
print("GNNFormer - Cross Validation")
print("=" * 80)

def gnnformer_fit_predict(X_tr, y_tr, X_va):

    ctor = lambda: GNNFormer(N_FEATURES, N_CLASSES)

    y_pred, y_proba, _ = train_torch_classifier(ctor, X_tr, y_tr, X_va)

    return y_pred, y_proba

gnnformer_oof_pred, gnnformer_oof_proba, gnnformer_fold_records = fit_folds(MODEL_NAME, gnnformer_fit_predict)

if CONFIG["use_smote_per_fold"]:

    smote_final = SMOTE(random_state=CONFIG["random_seed"])

    X_final, y_final = smote_final.fit_resample(X_cv_pool, y_cv_pool)

else:

    X_final, y_final = X_cv_pool, y_cv_pool

gnnformer_test_pred, gnnformer_test_proba, gnnformer_final = train_torch_classifier(
    lambda: GNNFormer(N_FEATURES, N_CLASSES), X_final, y_final, X_test,
)

torch.save(gnnformer_final.state_dict(), TRAINED_MODEL_DIR / "gnnformer_final.pt")

print("Saved:", TRAINED_MODEL_DIR / "gnnformer_final.pt")

cleanup_torch()
print_ram_usage("after cell 21")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### GNNFormer -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- GNNFormer (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "gnnformer"

if CONFIG["run_validation"]:

    gnnformer_cv_metrics = score_folds(MODEL_NAME, gnnformer_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    gnnformer_cv_metrics = None

print()

gnnformer_test_metrics = evaluate_on_test(MODEL_NAME, gnnformer_test_pred, gnnformer_test_proba)


## Model 6 — GraphTransformer (feature-graph, learned attention adjacency)

A cleaner "graph transformer" baseline distinct from GNNFormer: instead of a
GCN layer, feature-nodes attend to each other with plain multi-head
self-attention where attention weights ARE the (soft) graph edges — the
standard "Graph Transformer" formulation (attention-as-adjacency), with no
separate message-passing stage.

In [ ]:
# ==============================================================================
# Cell 15 : Model 6 - GraphTransformer (attention-as-adjacency)
# ==============================================================================
# Depends only on: Cells 1-10.
# ==============================================================================

class GraphTransformer(nn.Module):

    def __init__(self, n_features, n_classes, d_model=32, n_layers=3, n_heads=4, dropout=0.1):

        super().__init__()

        self.token_embed = nn.Parameter(torch.randn(n_features, d_model) * 0.02)

        self.pos_embed = nn.Parameter(torch.randn(1, n_features, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )

        self.graph_attn_layers = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.readout_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):

        # Network flow features (duration, bytes/s, etc.) are heavy-tailed --
        # even after StandardScaler, extreme outliers can have huge z-scores.
        # Multiplying that directly into a learned per-feature token (below)
        # can overflow to Inf/NaN inside the transformer's softmax/LayerNorm,
        # which is exactly what caused every epoch to report NaN/Inf loss.
        # Clamping bounds the input without materially changing its meaning
        # (+/-10 std devs is already far into the outlier tail).
        x = torch.clamp(x, -10.0, 10.0)

        # feature-nodes: value * learned per-feature token, plus a
        # positional embedding identifying WHICH feature the node represents
        node_feats = x.unsqueeze(-1) * self.token_embed.unsqueeze(0) + self.pos_embed

        # self-attention over feature-nodes == soft graph edges (Graph Transformer)
        encoded = self.graph_attn_layers(node_feats)

        pooled = self.readout_norm(encoded.mean(dim=1))

        return self.head(pooled)


# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("graphtransformer")

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in TRAIN_FEATURES_FILE.lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)

_print_training_data_summary("graphtransformer")

MODEL_NAME = "graphtransformer"

print("=" * 80)
print("GraphTransformer - Cross Validation")
print("=" * 80)

def graphtransformer_fit_predict(X_tr, y_tr, X_va):

    ctor = lambda: GraphTransformer(N_FEATURES, N_CLASSES)

    y_pred, y_proba, _ = train_torch_classifier(ctor, X_tr, y_tr, X_va)

    return y_pred, y_proba

graphtransformer_oof_pred, graphtransformer_oof_proba, graphtransformer_fold_records = fit_folds(MODEL_NAME, graphtransformer_fit_predict)

if CONFIG["use_smote_per_fold"]:

    smote_final = SMOTE(random_state=CONFIG["random_seed"])

    X_final, y_final = smote_final.fit_resample(X_cv_pool, y_cv_pool)

else:

    X_final, y_final = X_cv_pool, y_cv_pool

graphtransformer_test_pred, graphtransformer_test_proba, graphtransformer_final = train_torch_classifier(
    lambda: GraphTransformer(N_FEATURES, N_CLASSES), X_final, y_final, X_test,
)

torch.save(graphtransformer_final.state_dict(), TRAINED_MODEL_DIR / "graphtransformer_final.pt")

print("Saved:", TRAINED_MODEL_DIR / "graphtransformer_final.pt")

cleanup_torch()
print_ram_usage("after cell 23")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### GraphTransformer -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- GraphTransformer (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "graphtransformer"

if CONFIG["run_validation"]:

    graphtransformer_cv_metrics = score_folds(MODEL_NAME, graphtransformer_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    graphtransformer_cv_metrics = None

print()

graphtransformer_test_metrics = evaluate_on_test(MODEL_NAME, graphtransformer_test_pred, graphtransformer_test_proba)


### Configure TabPFN API Token

To use TabPFN in a non-interactive environment like Colab, you need to provide an API token. Follow these steps to obtain and configure your token:

1.  **Get your API Key:**
    *   Open `https://ux.priorlabs.ai` in your browser.
    *   Log in (or register if you don't have an account).
    *   Accept the license on the 'Licenses' tab.
    *   Copy your API Key from the 'Account' page (`https://ux.priorlabs.ai/account`).

2.  **Store the API Key in Colab Secrets:**
    *   In the left sidebar of your Colab notebook, click the "🔑" (Secrets) icon.
    *   Click `+ New secret`.
    *   For the "Name" field, enter `TABPFN_TOKEN`.
    *   For the "Value" field, paste your API Key copied from `https://ux.priorlabs.ai/account`.
    *   Ensure the "Notebook access" toggle is enabled for this secret.

In [ ]:
# TabPFN authentication: use an environment variable locally or a Colab
# Secret named TABPFN_TOKEN. Never paste a token into a notebook cell because
# notebook source is commonly committed or shared.
import os

if not os.environ.get("TABPFN_TOKEN") and RUN_ENVIRONMENT == "colab":
    try:
        from google.colab import userdata
        secret = userdata.get("TABPFN_TOKEN")
        if secret:
            os.environ["TABPFN_TOKEN"] = secret
    except Exception as exc:
        print(f"Could not read the Colab TABPFN_TOKEN secret: {exc}")

if os.environ.get("TABPFN_TOKEN"):
    print("TabPFN authentication is configured.")
else:
    print("TabPFN authentication is not configured; Model 7 may require it.")


## Model 7 — TabPFN-2.5 (2025)

TabPFN is a prior-fitted (in-context learning) transformer: it does not
"train" in the usual sense, it conditions on the (small) training set at
inference time. It currently supports bounded rows/features, so training
data is subsampled per fold (`TABPFN_MAX_TRAIN_ROWS`, `TABPFN_MAX_FEATURES`
in the **Master Control Cell**) — this is disclosed explicitly in the
results table so it is a fair, documented comparison rather than a hidden
shortcut.

In [ ]:
# ==============================================================================
# Cell 16 : Model 7 - TabPFN-2.5
# ==============================================================================
# Depends only on: Cells 1-9 (does not need the shared PyTorch trainer).
#
# NOTE: TabPFN requires a ONE-TIME license acceptance before it can download
# its pretrained model weights. Since Colab has no interactive browser
# prompt, you must supply an API token the first time you run this cell
# (or the first time in a fresh Colab runtime).
#   1. Go to https://ux.priorlabs.ai and log in / register (free)
#   2. Accept the license on the "Licenses" tab
#   3. Copy your API key from https://ux.priorlabs.ai/account
#   4. Set it as the TABPFN_TOKEN environment variable BEFORE launching
#      Jupyter (or as a Colab Secret). Never paste it into this cell.
# ==============================================================================

import os
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------------
# TabPFN License / Authentication
# ------------------------------------------------------------------
TABPFN_TOKEN = os.environ.get("TABPFN_TOKEN", "")

if TABPFN_TOKEN:
    os.environ["TABPFN_TOKEN"] = TABPFN_TOKEN
    print("TabPFN token read from the environment.")
elif "TABPFN_TOKEN" in os.environ:
    print("TabPFN token already present in environment (from a previous cell/session).")
else:
    print("WARNING: TABPFN_TOKEN is empty and not set in the environment.")
    print("Set TABPFN_TOKEN as a Colab Secret or local environment variable.")

from tabpfn import TabPFNClassifier

# ==============================================================================
# Print exactly what data THIS model is about to train/evaluate on
# ==============================================================================
def _print_training_data_summary(model_name):
    print("=" * 80)
    print(f"[{model_name}] DATA USED FOR TRAINING / CV / TEST")
    print("=" * 80)

    print(f"Loaded from files:")
    print(f"  Train X : {TRAIN_FEATURES_FILE}   ({X_train.shape[0]:,} rows, {X_train.shape[1]} cols)")
    print(f"  Val   X : {VAL_FEATURES_FILE}   ({X_val.shape[0]:,} rows)")
    print(f"  Test  X : {TEST_FEATURES_FILE}   ({X_test.shape[0]:,} rows)")

    print(f"\nCV_ON = '{CONFIG['cv_on']}'  ->  X_cv_pool shape: {X_cv_pool.shape}")
    print(f"USE_SMOTE_PER_FOLD = {CONFIG['use_smote_per_fold']}")
    print(f"N_FOLDS = {CONFIG['n_folds']}   STRATIFIED = {CONFIG['stratified']}")

    counts = pd.Series(y_cv_pool).value_counts().sort_index()
    dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(counts.index),
        "Count": counts.values,
        "Percent": (counts.values / counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    print(f"\nX_cv_pool class distribution ({counts.sum():,} rows total):")
    display(dist)

    if "smote" in str(TRAIN_FEATURES_FILE).lower():
        print(f"NOTE: TRAIN_FEATURES_FILE contains SMOTE-synthetic rows — "
              f"the counts above include synthetic data, not just real traffic.")

    print(f"\nHeld-out TEST set class distribution (real, untouched, {y_test.shape[0]:,} rows):")
    test_counts = pd.Series(y_test).value_counts().sort_index()
    test_dist = pd.DataFrame({
        "Class": label_encoder.inverse_transform(test_counts.index),
        "Count": test_counts.values,
        "Percent": (test_counts.values / test_counts.sum() * 100).round(2),
    }).sort_values("Count", ascending=False).reset_index(drop=True)
    display(test_dist)
    print("=" * 80)


_print_training_data_summary("tabpfn")

MODEL_NAME = "tabpfn"

print("=" * 80)
print("TabPFN-2.5 - Cross Validation & Inference")
print("=" * 80)
print(f"NOTE: subsampling to <= {CONFIG['tabpfn_max_train_rows']} rows, "
      f"<= {CONFIG['tabpfn_max_features']} features per fold (TabPFN constraint).")

from sklearn.model_selection import train_test_split

def subsample_for_tabpfn(X, y):
    if len(X) > CONFIG["tabpfn_max_train_rows"]:
        X_sub, _, y_sub, _ = train_test_split(
            X, y, train_size=CONFIG["tabpfn_max_train_rows"],
            stratify=y, random_state=CONFIG["random_seed"],
        )
    else:
        X_sub, y_sub = X, y

    if X_sub.shape[1] > CONFIG["tabpfn_max_features"]:
        X_sub = X_sub.iloc[:, :CONFIG["tabpfn_max_features"]]

    return X_sub, y_sub


def predict_proba_batched(model, X_data, batch_size=2000):
    """Run inference in small batches so we never exceed local GPU VRAM.
    TabPFN's attention scales with test-set size, so passing 250K rows in
    one call (fine on a Colab T4) can OOM on an 8GB local card."""
    probas = []
    n_samples = len(X_data)
    for i in tqdm(range(0, n_samples, batch_size), desc="Predicting in batches"):
        batch = X_data.iloc[i: i + batch_size]
        probas.append(model.predict_proba(batch))
    return np.vstack(probas)


def tabpfn_fit_predict(X_tr, y_tr, X_va):
    X_tr_sub, y_tr_sub = subsample_for_tabpfn(X_tr, y_tr)
    X_va_sub = X_va.iloc[:, :X_tr_sub.shape[1]]

    model = TabPFNClassifier(device=CONFIG["device_preference"])
    model.fit(X_tr_sub, y_tr_sub)

    y_proba = predict_proba_batched(model, X_va_sub, batch_size=2000)
    y_pred = y_proba.argmax(axis=1)

    return y_pred, y_proba


tabpfn_oof_pred, tabpfn_oof_proba, tabpfn_fold_records = fit_folds(MODEL_NAME, tabpfn_fit_predict)

# ==============================================================================
# Final Fit -> Held-Out Test Evaluation
# ==============================================================================

X_final_sub, y_final_sub = subsample_for_tabpfn(X_cv_pool, y_cv_pool)
X_test_sub = X_test.iloc[:, :X_final_sub.shape[1]]

tabpfn_final = TabPFNClassifier(device=CONFIG["device_preference"])
tabpfn_final.fit(X_final_sub, y_final_sub)

print("Running batched inference on Held-out Test Set...")
tabpfn_test_proba = predict_proba_batched(tabpfn_final, X_test_sub, batch_size=2000)
tabpfn_test_pred = tabpfn_test_proba.argmax(axis=1)

import joblib as _joblib
_joblib.dump(tabpfn_final, TRAINED_MODEL_DIR / "tabpfn_final.pkl")
print("Saved:", TRAINED_MODEL_DIR / "tabpfn_final.pkl")

cleanup_torch()
print_ram_usage("after cell 16")

print()
print(f"[{MODEL_NAME}] Training complete. Run the next cell (VALIDATION) to score it.")


### TabPFN-2.5 -- Validation (CV scoring + Test scoring)

In [ ]:
# ==============================================================================
# VALIDATION -- TabPFN-2.5 (CV fold scoring + held-out TEST scoring)
# CV fold scoring only runs if CONFIG["run_validation"] is True; the held-out
# TEST score always runs -- it is the final reported result regardless of the
# validation/K-fold toggles.
# ==============================================================================

MODEL_NAME = "tabpfn"

if CONFIG["run_validation"]:

    tabpfn_cv_metrics = score_folds(MODEL_NAME, tabpfn_fold_records)

else:

    print(f"[{MODEL_NAME}] RUN_VALIDATION=False -> CV fold scoring skipped.")

    tabpfn_cv_metrics = None

print()

tabpfn_test_metrics = evaluate_on_test(MODEL_NAME, tabpfn_test_pred, tabpfn_test_proba)


In [ ]:
# ==============================================================================
# Cell 18 : Model Comparison Table & Figures
# ==============================================================================
# Reads every "<model>_cv_fold_metrics.csv" and "<model>_test_metrics.csv"
# saved by the model cells above. Run this AFTER at least 2 model cells.
# Depends only on: Cells 1-8 (re-reads everything from disk).
# ==============================================================================

import glob

print("=" * 80)
print("Aggregating All Model Results")
print("=" * 80)

cv_files = sorted(glob.glob(str(TABLE_DIR / "*_cv_fold_metrics.csv")))

all_cv = pd.concat([pd.read_csv(f) for f in cv_files], ignore_index=True)

print("Models found (CV):", all_cv["Model"].unique().tolist())

# ------------------------------------------------------------------
# CV summary: mean +/- std per model
# ------------------------------------------------------------------
cv_summary = all_cv.groupby("Model").agg(["mean", "std"])

cv_summary.to_csv(TABLE_DIR / "all_models_cv_summary.csv")

display(cv_summary)

# ------------------------------------------------------------------
# Held-out test summary
# ------------------------------------------------------------------
test_files = sorted(glob.glob(str(TABLE_DIR / "*_test_metrics.csv")))

test_rows = []

for f in test_files:

    model_name = Path(f).name.replace("_test_metrics.csv", "")

    row = pd.read_csv(f).iloc[0].to_dict()

    row["Model"] = model_name

    test_rows.append(row)

all_test = pd.DataFrame(test_rows).set_index("Model").sort_values("f1_macro", ascending=False)

all_test.to_csv(TABLE_DIR / "all_models_test_summary.csv")

display(all_test)

# ------------------------------------------------------------------
# Bar chart: F1-macro (test) per model
# ------------------------------------------------------------------
plt.figure(figsize=(12, 6))

sns.barplot(x=all_test.index, y=all_test["f1_macro"])

plt.xticks(rotation=30, ha="right")

plt.title("Test-Set F1-Macro by Model", fontsize=16)

plt.ylabel("F1-Macro")

plt.tight_layout()

plt.savefig(FIGURE_DIR / f"all_models_f1_macro.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])

plt.show()

# ------------------------------------------------------------------
# Bar chart: Accuracy (test) per model
# ------------------------------------------------------------------
plt.figure(figsize=(12, 6))

sns.barplot(x=all_test.index, y=all_test["accuracy"])

plt.xticks(rotation=30, ha="right")

plt.title("Test-Set Accuracy by Model", fontsize=16)

plt.ylabel("Accuracy")

plt.tight_layout()

plt.savefig(FIGURE_DIR / f"all_models_accuracy.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])

plt.show()

# ------------------------------------------------------------------
# CV box plots: F1-macro spread per fold, per model
# ------------------------------------------------------------------
plt.figure(figsize=(13, 7))

sns.boxplot(data=all_cv, x="Model", y="f1_macro")

sns.stripplot(data=all_cv, x="Model", y="f1_macro", color="black", alpha=0.5, size=4)

plt.xticks(rotation=30, ha="right")

plt.title(f"{CONFIG['n_folds']}-Fold CV F1-Macro Distribution by Model", fontsize=16)

plt.tight_layout()

plt.savefig(FIGURE_DIR / f"all_models_cv_boxplot.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])

plt.show()

# ------------------------------------------------------------------
# Radar / multi-metric comparison table (rendered, not just saved)
# ------------------------------------------------------------------
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "mcc"]

print()
print("=" * 80)
print("Final Multi-Metric Comparison (Held-Out Test Set)")
print("=" * 80)

display(all_test[metric_cols].round(4))

cleanup_torch()
print_ram_usage("after cell 32")


## Hybrid Model — Stacked Ensemble (2–3 Best Models)

Combines the members listed in `HYBRID_MEMBERS` (Master Control Cell) using
either **stacking** (a logistic-regression meta-learner trained on the
out-of-fold predictions of the base models — leakage-safe because OOF
predictions are used) or **soft voting** (simple probability averaging).

In [ ]:
# ==============================================================================
# Cell 17 : Hybrid / Stacked Ensemble (Auto-Selected or Manual)
# ==============================================================================
# Auto-selection ranks individual models by mean cross-validation score.
# The held-out test set is never used to select ensemble members.
# ==============================================================================

from sklearn.linear_model import LogisticRegression

print("=" * 80)
print("Selecting Hybrid Members")
print("=" * 80)

if CONFIG["hybrid_auto_select"]:

    cv_files = sorted(TABLE_DIR.glob("*_cv_fold_metrics.csv"))
    if not cv_files:
        raise FileNotFoundError(
            "No CV fold metrics found. Run at least the requested number of "
            "individual model validation cells before auto-selecting the hybrid."
        )

    cv_scores = pd.concat([pd.read_csv(path) for path in cv_files], ignore_index=True)
    metric = CONFIG["hybrid_select_metric"]
    if metric not in cv_scores.columns:
        raise KeyError(f"Hybrid selection metric '{metric}' is absent from CV metrics.")

    ranking = cv_scores.groupby("Model")[metric].agg(["mean", "std"])
    ranking = ranking[~ranking.index.str.startswith("hybrid_")]
    ranking = ranking.sort_values("mean", ascending=False)

    if len(ranking) < CONFIG["hybrid_top_n"]:
        raise ValueError(
            f"Need {CONFIG['hybrid_top_n']} completed individual models, "
            f"but found only {len(ranking)} in CV metrics."
        )

    hybrid_members = ranking.head(CONFIG["hybrid_top_n"]).index.tolist()

    print(f"Auto-selected top {CONFIG['hybrid_top_n']} models by mean CV '{metric}':")
    display(ranking.head(CONFIG["hybrid_top_n"]))

else:

    hybrid_members = CONFIG["hybrid_members_manual"]

    print("Manual hybrid members (HYBRID_AUTO_SELECT=False):", hybrid_members)

MODEL_NAME = "hybrid_" + "_".join(hybrid_members)

print()
print("=" * 80)
print(f"Hybrid Model ({CONFIG['hybrid_strategy']}) : {hybrid_members}")
print("=" * 80)

# ------------------------------------------------------------------
# Load test probabilities for each member (always required).
# OOF probabilities are only loaded if we actually end up stacking --
# checking file existence FIRST avoids crashing on a missing OOF file
# when we would have fallen back to soft_voting anyway.
# ------------------------------------------------------------------
test_proba_list = []
missing_test = []

for member in hybrid_members:
    test_path = PRED_DIR / f"{member}_test_proba.npy"
    if not test_path.exists():
        missing_test.append(member)
        continue
    test_proba_list.append(np.load(test_path))
    print(f"Loaded test probabilities for: {member}")

if missing_test:
    raise FileNotFoundError(
        f"Missing test-set probability file(s) for: {missing_test}. "
        "Run each member's TRAIN cell (and its evaluate_on_test call) before hybrid."
    )

oof_missing = [
    m for m in hybrid_members
    if not (PRED_DIR / f"{m}_oof_proba.npy").exists()
]

if CONFIG["hybrid_strategy"] == "stacking" and (not CONFIG["run_validation"] or oof_missing):

    if oof_missing:
        print(f"NOTE: missing OOF probability file(s) for: {oof_missing}.")
    print("NOTE: hybrid_strategy='stacking' needs each member's OOF probabilities, "
          "which are only produced when RUN_VALIDATION=True (fit_folds saves them). "
          "Falling back to 'soft_voting' for this run -- set RUN_VALIDATION=True and "
          "re-run the member TRAIN cells to enable stacking again.")

    hybrid_strategy_effective = "soft_voting"

else:

    hybrid_strategy_effective = CONFIG["hybrid_strategy"]

if hybrid_strategy_effective == "soft_voting":

    hybrid_test_proba = np.mean(test_proba_list, axis=0)

    hybrid_test_pred = hybrid_test_proba.argmax(axis=1)

else:  # stacking

    oof_proba_list = [np.load(PRED_DIR / f"{m}_oof_proba.npy") for m in hybrid_members]

    meta_X_train = np.concatenate(oof_proba_list, axis=1)

    meta_X_test = np.concatenate(test_proba_list, axis=1)

    meta_model = LogisticRegression(max_iter=2000, random_state=CONFIG["random_seed"])

    meta_model.fit(meta_X_train, y_cv_pool)

    hybrid_test_proba = meta_model.predict_proba(meta_X_test)

    hybrid_test_pred = meta_model.predict(meta_X_test)

    joblib.dump(meta_model, TRAINED_MODEL_DIR / f"{MODEL_NAME}_meta_model.pkl")

hybrid_test_metrics = evaluate_on_test(MODEL_NAME, hybrid_test_pred, hybrid_test_proba)

print()
print("Hybrid strategy   :", hybrid_strategy_effective)
print("Members combined  :", hybrid_members)
print("Selection method  :", "auto" if CONFIG["hybrid_auto_select"] else "manual")

cleanup_torch()
print_ram_usage("after cell 30")


## Model Comparison — All Models, All Folds, Test Set

In [ ]:
# ==============================================================================
# Cell 18 : Model Comparison Table & Figures
# ==============================================================================
# Reads every "<model>_cv_fold_metrics.csv" and "<model>_test_metrics.csv"
# saved by the model cells above. Run this AFTER at least 2 model cells.
# Depends only on: Cells 1-8 (re-reads everything from disk).
# ==============================================================================

import glob

print("=" * 80)
print("Aggregating All Model Results")
print("=" * 80)

# ------------------------------------------------------------------
# CV summary (only if fold metric files exist -- they won't if every
# model in this run had RUN_VALIDATION=False)
# ------------------------------------------------------------------
cv_files = sorted(glob.glob(str(TABLE_DIR / "*_cv_fold_metrics.csv")))

if not cv_files:
    print("\n[NOTE] No '*_cv_fold_metrics.csv' files found.")
    print("       (Expected if RUN_VALIDATION=False was used for every trained model.)\n")
    all_cv = None
else:
    all_cv = pd.concat([pd.read_csv(f) for f in cv_files], ignore_index=True)
    print("Models found (CV):", all_cv["Model"].unique().tolist())

    cv_summary = all_cv.groupby("Model").agg(["mean", "std"])
    cv_summary.to_csv(TABLE_DIR / "all_models_cv_summary.csv")
    display(cv_summary)

# ------------------------------------------------------------------
# Held-out test summary
# ------------------------------------------------------------------
test_files = sorted(glob.glob(str(TABLE_DIR / "*_test_metrics.csv")))

if not test_files:
    raise FileNotFoundError(
        f"No '*_test_metrics.csv' files found in {TABLE_DIR}. "
        "Run at least one model's TRAIN cell before this comparison cell."
    )

test_rows = []
for f in test_files:
    model_name = Path(f).name.replace("_test_metrics.csv", "")
    row = pd.read_csv(f).iloc[0].to_dict()
    row["Model"] = model_name
    test_rows.append(row)

all_test = pd.DataFrame(test_rows).set_index("Model").sort_values("f1_macro", ascending=False)
all_test.to_csv(TABLE_DIR / "all_models_test_summary.csv")
print("\nModels found (Test Set):", all_test.index.tolist())
display(all_test)

# ------------------------------------------------------------------
# Bar chart: F1-macro (test) per model
# ------------------------------------------------------------------
plt.figure(figsize=(12, 6))
sns.barplot(x=all_test.index, y=all_test["f1_macro"])
plt.xticks(rotation=30, ha="right")
plt.title("Test-Set F1-Macro by Model", fontsize=16)
plt.ylabel("F1-Macro")
plt.tight_layout()
plt.savefig(FIGURE_DIR / f"all_models_f1_macro.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])
plt.show()

# ------------------------------------------------------------------
# Bar chart: Accuracy (test) per model
# ------------------------------------------------------------------
plt.figure(figsize=(12, 6))
sns.barplot(x=all_test.index, y=all_test["accuracy"])
plt.xticks(rotation=30, ha="right")
plt.title("Test-Set Accuracy by Model", fontsize=16)
plt.ylabel("Accuracy")
plt.tight_layout()
plt.savefig(FIGURE_DIR / f"all_models_accuracy.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])
plt.show()

# ------------------------------------------------------------------
# CV box plots: F1-macro spread per fold, per model (only if we have CV data)
# ------------------------------------------------------------------
if all_cv is not None and not all_cv.empty:
    plt.figure(figsize=(13, 7))
    sns.boxplot(data=all_cv, x="Model", y="f1_macro")
    sns.stripplot(data=all_cv, x="Model", y="f1_macro", color="black", alpha=0.5, size=4)
    plt.xticks(rotation=30, ha="right")
    plt.title(f"{CONFIG['n_folds']}-Fold CV F1-Macro Distribution by Model", fontsize=16)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"all_models_cv_boxplot.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])
    plt.show()
else:
    print("\n[Skipped] CV box plot -- no cross-validation fold data available.")

# ------------------------------------------------------------------
# Radar / multi-metric comparison table (rendered, not just saved)
# ------------------------------------------------------------------
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "mcc"]
available_cols = [c for c in metric_cols if c in all_test.columns]

print()
print("=" * 80)
print("Final Multi-Metric Comparison (Held-Out Test Set)")
print("=" * 80)

display(all_test[available_cols].round(4))

cleanup_torch()
print_ram_usage("after cell 18")


## Statistical Significance — Paired t-test & McNemar's Test

In [ ]:
# ==============================================================================
# Cell 19 : Paired t-test (Cross-Validation Fold Scores)
# ==============================================================================
# Tests whether the DIFFERENCE in per-fold F1-macro between every pair of
# models is statistically significant. Valid because every model used the
# SAME fold indices (Cell 7), so folds are paired observations.
# Depends only on: Cell 18's "all_cv" (re-loads it if missing).
# ==============================================================================

import glob
from scipy import stats
from itertools import combinations

print("=" * 80)
print("Paired t-test (per-fold F1-macro)")
print("=" * 80)

# "all_cv" may exist in globals() but be None (Cell 18 sets it to None when
# no CV fold files were found), so check for that explicitly too.
if "all_cv" not in globals() or all_cv is None:
    cv_files = sorted(glob.glob(str(TABLE_DIR / "*_cv_fold_metrics.csv")))
    all_cv = pd.concat([pd.read_csv(f) for f in cv_files], ignore_index=True) if cv_files else None

if all_cv is None or all_cv.empty:
    print("\n[NOTE] No cross-validation fold metrics available.")
    print("       Paired t-test skipped. Set RUN_VALIDATION=True during model runs to generate fold metrics.")
elif all_cv["Model"].nunique() < 2:
    print(f"\n[NOTE] Only {all_cv['Model'].nunique()} model has CV fold metrics.")
    print("       Paired t-test requires at least 2 models for comparison.")
else:
    pivot_f1 = all_cv.pivot(index="Fold", columns="Model", values="f1_macro")
    models_present = list(pivot_f1.columns)

    ttest_rows = []
    for model_a, model_b in combinations(models_present, 2):
        stat, p_value = stats.ttest_rel(pivot_f1[model_a], pivot_f1[model_b])
        ttest_rows.append({
            "Model A": model_a,
            "Model B": model_b,
            "Mean F1 A": pivot_f1[model_a].mean(),
            "Mean F1 B": pivot_f1[model_b].mean(),
            "t-statistic": stat,
            "p-value": p_value,
            "Significant (p<0.05)": p_value < 0.05,
        })

    ttest_df = pd.DataFrame(ttest_rows).sort_values("p-value")

    STATS_DIR.mkdir(parents=True, exist_ok=True)
    ttest_df.to_csv(STATS_DIR / "paired_ttest_results.csv", index=False)
    display(ttest_df)

    # ------------------------------------------------------------------
    # p-value heatmap (model x model)
    # ------------------------------------------------------------------
    p_matrix = pd.DataFrame(np.nan, index=models_present, columns=models_present)
    for row in ttest_rows:
        p_matrix.loc[row["Model A"], row["Model B"]] = row["p-value"]
        p_matrix.loc[row["Model B"], row["Model A"]] = row["p-value"]

    plt.figure(figsize=(9, 7))
    sns.heatmap(p_matrix.astype(float), annot=True, fmt=".3f", cmap="coolwarm_r", vmin=0, vmax=0.1)
    plt.title("Paired t-test p-values (F1-Macro, per fold)", fontsize=15)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"paired_ttest_heatmap.{CONFIG['figure_format']}", dpi=CONFIG["figure_dpi"])
    plt.show()

    print()
    print("Saved:", STATS_DIR / "paired_ttest_results.csv")

cleanup_torch()
print_ram_usage("after cell 19")


In [ ]:
# ==============================================================================
# Cell 20 : McNemar's Test (Held-Out Test Set Predictions)
# ==============================================================================
# Pairwise test of whether two models make DIFFERENT kinds of errors on the
# SAME held-out test instances (contingency of correct/incorrect).
# Depends only on: "<model>_test_predictions.csv" files saved by evaluate_on_test.
# ==============================================================================

from statsmodels.stats.contingency_tables import mcnemar

print("=" * 80)
print("McNemar's Test (Test-Set Predictions)")
print("=" * 80)

pred_files = sorted(glob.glob(str(PRED_DIR / "*_test_predictions.csv")))

model_preds = {}

for f in pred_files:

    model_name = Path(f).name.replace("_test_predictions.csv", "")

    df = pd.read_csv(f)

    model_preds[model_name] = df["y_pred"].values

print("Models found:", list(model_preds.keys()))

mcnemar_rows = []

for model_a, model_b in combinations(model_preds.keys(), 2):

    correct_a = (model_preds[model_a] == y_test)

    correct_b = (model_preds[model_b] == y_test)

    # 2x2 contingency table: [both correct, a correct/b wrong], [a wrong/b correct, both wrong]
    n_both_correct = int(np.sum(correct_a & correct_b))

    n_a_only = int(np.sum(correct_a & ~correct_b))

    n_b_only = int(np.sum(~correct_a & correct_b))

    n_both_wrong = int(np.sum(~correct_a & ~correct_b))

    table = [[n_both_correct, n_a_only], [n_b_only, n_both_wrong]]

    result = mcnemar(table, exact=(min(n_a_only, n_b_only) < 25), correction=True)

    mcnemar_rows.append({
        "Model A": model_a,
        "Model B": model_b,
        "A only correct": n_a_only,
        "B only correct": n_b_only,
        "statistic": result.statistic,
        "p-value": result.pvalue,
        "Significant (p<0.05)": result.pvalue < 0.05,
    })

mcnemar_df = pd.DataFrame(mcnemar_rows).sort_values("p-value")

mcnemar_df.to_csv(STATS_DIR / "mcnemar_test_results.csv", index=False)

display(mcnemar_df)

print()
print("Saved:", STATS_DIR / "mcnemar_test_results.csv")

cleanup_torch()
print_ram_usage("after cell 35")


## Explainable AI (XAI)

### XAI Helper — Unified `predict_proba()` for All 7 Models + Hybrid

Shared by the SHAP, Permutation Importance, and LIME cells below so all
three can loop over every base model **and** the hybrid ensemble without
duplicating model-loading logic. `load_xai_predict_fn(model_key)` returns a
single callable with the same signature regardless of what's underneath it
(CatBoost, a PyTorch model, TabPFN, or a combination of several via the
hybrid ensemble) — that's what lets the same SHAP/LIME/permutation code work
for all 8.

Depends on: Cells 1-9 (data/config) + the 5 PyTorch model class definitions
already defined earlier (Cells 18/22/26/30/34) + Cell 18's comparison table
(only if `hybrid_auto_select=True`, to know which models the hybrid uses).


In [ ]:
# ==============================================================================
# Cell 20b : XAI Helper — Unified predict_proba() for ANY model (incl. hybrid)
# ==============================================================================
from catboost import CatBoostClassifier as _XAI_CatBoost

TORCH_MODEL_CLASSES_XAI = {
    "fttransformer": FTTransformer,
    "tabm": TabM,
    "tabgt": TabGT,
    "gnnformer": GNNFormer,
    "graphtransformer": GraphTransformer,
}

XAI_DISPLAY_NAMES = {
    "catboost": "CatBoost",
    "fttransformer": "FT-Transformer",
    "tabm": "TabM",
    "tabgt": "TabGT",
    "gnnformer": "GNNFormer",
    "graphtransformer": "GraphTransformer",
    "tabpfn": "TabPFN-2.5",
    "hybrid": "Hybrid",
}


def _resolve_hybrid_members():
    """Same selection logic as Cell 17 (Hybrid) -- auto-select from the
    saved test-summary ranking, or fall back to the manual list. Needed so
    XAI explains the SAME hybrid combination that was actually trained."""
    if CONFIG["hybrid_auto_select"]:
        summary_path = TABLE_DIR / "all_models_test_summary.csv"
        if not summary_path.exists():
            raise FileNotFoundError(
                "all_models_test_summary.csv not found -- run Cell 18 "
                "(Comparison) at least once before requesting hybrid XAI."
            )
        ranking = pd.read_csv(summary_path, index_col=0)
        ranking = ranking[~ranking.index.str.startswith("hybrid_")]
        ranking = ranking.sort_values(CONFIG["hybrid_select_metric"], ascending=False)
        return ranking.head(CONFIG["hybrid_top_n"]).index.tolist()
    return CONFIG["hybrid_members_manual"]


def load_xai_predict_fn(model_key):
    """Returns (predict_proba_fn, display_name) for any of the 7 base models
    or 'hybrid'. predict_proba_fn takes a DataFrame/ndarray of raw feature
    rows and returns an (n_rows, N_CLASSES) probability array -- the same
    interface SHAP's KernelExplainer, permutation importance, and LIME all
    expect, which is what lets one function serve all three XAI cells."""

    device = get_device()
    display_name = XAI_DISPLAY_NAMES.get(model_key, model_key)

    if model_key == "catboost":
        m = _XAI_CatBoost()
        m.load_model(str(TRAINED_MODEL_DIR / "catboost_final.cbm"))
        return (lambda X: m.predict_proba(X)), display_name

    if model_key == "tabpfn":
        m = joblib.load(TRAINED_MODEL_DIR / "tabpfn_final.pkl")
        n_cols = min(N_FEATURES, CONFIG.get("tabpfn_max_features", N_FEATURES))

        def _predict_tabpfn(X, _m=m, _n=n_cols):
            X_df = X if hasattr(X, "iloc") else pd.DataFrame(X, columns=FEATURE_NAMES)
            return _m.predict_proba(X_df.iloc[:, :_n])

        return _predict_tabpfn, display_name

    if model_key in TORCH_MODEL_CLASSES_XAI:
        cls = TORCH_MODEL_CLASSES_XAI[model_key]
        m = cls(N_FEATURES, N_CLASSES).to(device)
        m.load_state_dict(torch.load(TRAINED_MODEL_DIR / f"{model_key}_final.pt", map_location=device))
        m.eval()

        def _predict_torch(X, _m=m, _device=device):
            X_vals = X.values if hasattr(X, "values") else X
            x_t = torch.tensor(X_vals, dtype=torch.float32).to(_device)
            with torch.no_grad():
                logits = _m(x_t)
            return torch.softmax(logits, dim=1).cpu().numpy()

        return _predict_torch, display_name

    if model_key == "hybrid":
        members = _resolve_hybrid_members()
        member_fns = {mem: load_xai_predict_fn(mem)[0] for mem in members}

        strategy = CONFIG["hybrid_strategy"]
        meta_model = None

        if strategy == "stacking":
            meta_path = TRAINED_MODEL_DIR / f"hybrid_{'_'.join(members)}_meta_model.pkl"
            if meta_path.exists():
                meta_model = joblib.load(meta_path)
            else:
                print(f"  NOTE: {meta_path.name} not found -- falling back to "
                      f"soft_voting for hybrid XAI.")
                strategy = "soft_voting"

        def _predict_hybrid(X, _fns=member_fns, _strategy=strategy, _meta=meta_model):
            member_probas = [fn(X) for fn in _fns.values()]
            if _strategy == "soft_voting":
                return np.mean(member_probas, axis=0)
            meta_X = np.concatenate(member_probas, axis=1)
            return _meta.predict_proba(meta_X)

        hybrid_display = "Hybrid (" + "+".join(members) + ")"
        return _predict_hybrid, hybrid_display

    raise ValueError(f"Unknown model_key for XAI: '{model_key}'")


print("XAI helper ready. XAI_MODELS =", CONFIG["xai_models"])


In [ ]:
# ==============================================================================
# Cell 21 : XAI - SHAP (Global + Local Explanations) — ALL Models + Hybrid
# ==============================================================================
# Loops over CONFIG["xai_models"] (default: all 7 base models + hybrid).
# CatBoost uses shap.TreeExplainer (exact, fast); every other model
# (including hybrid) uses shap.KernelExplainer wrapped around the unified
# predict_proba_fn from the XAI Helper cell above (model-agnostic, sampled).
# A failure on any one model (e.g. TabPFN token missing) is skipped with a
# warning rather than aborting the whole loop.
# Depends on: Cells 1-9 + the XAI Helper cell (Cell 20b) above.
# ==============================================================================

import shap

xai_sample = X_test.sample(
    n=min(CONFIG["xai_sample_size"], len(X_test)),
    random_state=CONFIG["random_seed"],
)

shap_status = []

for model_key in CONFIG["xai_models"]:

    print("=" * 80)
    print(f"SHAP Explanations : {model_key}")
    print("=" * 80)

    try:
        predict_fn, display_name = load_xai_predict_fn(model_key)

        if model_key == "catboost":
            cb_model = _XAI_CatBoost()
            cb_model.load_model(str(TRAINED_MODEL_DIR / "catboost_final.cbm"))
            explainer = shap.TreeExplainer(cb_model)
            shap_values = explainer.shap_values(xai_sample)
        else:
            background = shap.sample(xai_sample, 50, random_state=CONFIG["random_seed"])
            explainer = shap.KernelExplainer(predict_fn, background)
            shap_values = explainer.shap_values(xai_sample.iloc[:100], nsamples=100)

        # --------------------------------------------------------
        # Global summary plot (bar) - top-K features
        # --------------------------------------------------------
        plt.figure(figsize=(10, 8))
        try:
            shap.summary_plot(
                shap_values, xai_sample, feature_names=FEATURE_NAMES,
                max_display=CONFIG["xai_top_features"], plot_type="bar", show=False,
            )
        except Exception as e:
            print("  Summary bar plot fallback (multi-class averaging):", e)
        plt.title(f"SHAP Feature Importance — {display_name}", fontsize=13)
        plt.tight_layout()
        plt.savefig(XAI_DIR / f"shap_summary_bar_{model_key}.{CONFIG['figure_format']}",
                    dpi=CONFIG["figure_dpi"], bbox_inches="tight")
        plt.show()

        # --------------------------------------------------------
        # Global summary plot (beeswarm)
        # --------------------------------------------------------
        plt.figure(figsize=(10, 8))
        try:
            shap_for_beeswarm = shap_values[0] if isinstance(shap_values, list) else shap_values
            shap.summary_plot(
                shap_for_beeswarm, xai_sample, feature_names=FEATURE_NAMES,
                max_display=CONFIG["xai_top_features"], show=False,
            )
        except Exception as e:
            print("  Beeswarm plot skipped:", e)
        plt.title(f"SHAP Beeswarm — {display_name}", fontsize=13)
        plt.tight_layout()
        plt.savefig(XAI_DIR / f"shap_summary_beeswarm_{model_key}.{CONFIG['figure_format']}",
                    dpi=CONFIG["figure_dpi"], bbox_inches="tight")
        plt.show()

        print(f"  Saved SHAP figures for {display_name} to: {XAI_DIR}")
        shap_status.append({"model": model_key, "display_name": display_name, "status": "ok"})

    except Exception as e:
        print(f"  SKIPPED {model_key}: {type(e).__name__}: {e}")
        shap_status.append({"model": model_key, "display_name": model_key, "status": f"error: {e}"})

    finally:
        cleanup_torch()
        print_ram_usage(f"after SHAP [{model_key}]")

    print()

shap_status_df = pd.DataFrame(shap_status)
shap_status_df.to_csv(XAI_DIR / "shap_run_status.csv", index=False)
display(shap_status_df)
print("Done. Status saved to:", XAI_DIR / "shap_run_status.csv")


In [ ]:
# ==============================================================================
# Cell 22 : XAI - Permutation Importance — ALL Models + Hybrid
# ==============================================================================
# sklearn's permutation_importance() expects a real estimator object, which
# the hybrid ensemble and the raw PyTorch models don't provide -- so this
# uses a small custom permutation-importance loop built directly on top of
# the same unified predict_fn from the XAI Helper cell, giving identical
# results to sklearn's approach (shuffle a column, measure the F1-macro
# drop) but working for all 8 models with one code path.
# Depends on: Cells 1-9 + the XAI Helper cell (Cell 20b) above.
# ==============================================================================

from sklearn.metrics import f1_score


def permutation_importance_custom(predict_fn, X, y, n_repeats, random_state):
    rng = np.random.RandomState(random_state)

    baseline_pred = predict_fn(X).argmax(axis=1)
    baseline_score = f1_score(y, baseline_pred, average="macro")

    n_features_local = X.shape[1]
    importances = np.zeros((n_repeats, n_features_local))

    for col_i, col in enumerate(X.columns):
        for r in range(n_repeats):
            X_permuted = X.copy()
            X_permuted[col] = rng.permutation(X_permuted[col].values)
            permuted_pred = predict_fn(X_permuted).argmax(axis=1)
            permuted_score = f1_score(y, permuted_pred, average="macro")
            importances[r, col_i] = baseline_score - permuted_score

    return importances.mean(axis=0), importances.std(axis=0), baseline_score


perm_sample_X = X_test.sample(n=min(CONFIG["xai_sample_size"], len(X_test)), random_state=CONFIG["random_seed"])
perm_sample_y = y_test[perm_sample_X.index]

perm_status = []

for model_key in CONFIG["xai_models"]:

    print("=" * 80)
    print(f"Permutation Importance : {model_key}")
    print("=" * 80)

    try:
        predict_fn, display_name = load_xai_predict_fn(model_key)

        imp_mean, imp_std, baseline_f1 = permutation_importance_custom(
            predict_fn, perm_sample_X, perm_sample_y,
            n_repeats=CONFIG["perm_n_repeats"], random_state=CONFIG["random_seed"],
        )

        perm_df = pd.DataFrame({
            "Feature": FEATURE_NAMES,
            "Importance Mean": imp_mean,
            "Importance Std": imp_std,
        }).sort_values("Importance Mean", ascending=False)

        perm_df.to_csv(XAI_DIR / f"permutation_importance_{model_key}.csv", index=False)
        display(perm_df.head(CONFIG["xai_top_features"]))

        top_perm = perm_df.head(CONFIG["xai_top_features"])
        plt.figure(figsize=(10, 8))
        plt.barh(top_perm["Feature"][::-1], top_perm["Importance Mean"][::-1],
                 xerr=top_perm["Importance Std"][::-1])
        plt.xlabel("Decrease in F1-Macro when feature is permuted")
        plt.title(f"Permutation Importance — {display_name} (Top {CONFIG['xai_top_features']})", fontsize=14)
        plt.tight_layout()
        plt.savefig(XAI_DIR / f"permutation_importance_{model_key}.{CONFIG['figure_format']}",
                    dpi=CONFIG["figure_dpi"])
        plt.show()

        print(f"  Baseline F1-macro : {baseline_f1:.4f}")
        print(f"  Saved: {XAI_DIR / f'permutation_importance_{model_key}.csv'}")
        perm_status.append({"model": model_key, "display_name": display_name,
                             "status": "ok", "baseline_f1_macro": round(baseline_f1, 4)})

    except Exception as e:
        print(f"  SKIPPED {model_key}: {type(e).__name__}: {e}")
        perm_status.append({"model": model_key, "display_name": model_key, "status": f"error: {e}"})

    finally:
        cleanup_torch()
        print_ram_usage(f"after Permutation Importance [{model_key}]")

    print()

perm_status_df = pd.DataFrame(perm_status)
perm_status_df.to_csv(XAI_DIR / "permutation_importance_run_status.csv", index=False)
display(perm_status_df)
print("Done. Status saved to:", XAI_DIR / "permutation_importance_run_status.csv")


In [ ]:
# ==============================================================================
# Cell 23 : XAI - LIME (Local Explanation) — ALL Models + Hybrid
# ==============================================================================
# Explains the SAME test instance (row CONFIG["lime_instance_index"]) for
# every model in CONFIG["xai_models"], so you can directly compare how each
# model justifies (or fails to justify) its prediction on one concrete flow
# -- useful as a side-by-side case-study figure in the report/defense.
# Depends on: Cells 1-9 + the XAI Helper cell (Cell 20b) above.
# ==============================================================================

from lime.lime_tabular import LimeTabularExplainer

lime_background = X_train.sample(
    n=min(5000, len(X_train)), random_state=CONFIG["random_seed"],
) if len(X_train) > 5000 else X_train

instance_idx = CONFIG["lime_instance_index"]
instance = X_test.iloc[instance_idx].values
true_label = CLASS_NAMES[y_test[instance_idx]]

print(f"Explaining test instance #{instance_idx}  (True label: {true_label})")
print()

lime_status = []

for model_key in CONFIG["xai_models"]:

    print("=" * 80)
    print(f"LIME Explanation : {model_key}")
    print("=" * 80)

    try:
        predict_fn, display_name = load_xai_predict_fn(model_key)

        lime_explainer = LimeTabularExplainer(
            training_data=lime_background.values,
            feature_names=FEATURE_NAMES,
            class_names=CLASS_NAMES,
            mode="classification",
            random_state=CONFIG["random_seed"],
        )

        lime_explanation = lime_explainer.explain_instance(
            instance, predict_fn,
            num_features=CONFIG["xai_top_features"],
            num_samples=CONFIG["lime_num_samples"],
        )

        fig = lime_explanation.as_pyplot_figure()
        fig.set_size_inches(10, 7)
        plt.title(f"LIME Explanation — {display_name} — Instance #{instance_idx} (True: {true_label})", fontsize=12)
        plt.tight_layout()
        plt.savefig(XAI_DIR / f"lime_instance_{instance_idx}_{model_key}.{CONFIG['figure_format']}",
                    dpi=CONFIG["figure_dpi"])
        plt.show()

        lime_explanation.save_to_file(str(XAI_DIR / f"lime_instance_{instance_idx}_{model_key}.html"))

        fig_ext = CONFIG["figure_format"]
        print(f"  Saved: {XAI_DIR / f'lime_instance_{instance_idx}_{model_key}.html'}")
        print(f"  Saved: {XAI_DIR / f'lime_instance_{instance_idx}_{model_key}.{fig_ext}'}")
        lime_status.append({"model": model_key, "display_name": display_name, "status": "ok"})

    except Exception as e:
        print(f"  SKIPPED {model_key}: {type(e).__name__}: {e}")
        lime_status.append({"model": model_key, "display_name": model_key, "status": f"error: {e}"})

    finally:
        cleanup_torch()
        print_ram_usage(f"after LIME [{model_key}]")

    print()

lime_status_df = pd.DataFrame(lime_status)
lime_status_df.to_csv(XAI_DIR / "lime_run_status.csv", index=False)
display(lime_status_df)
print("Done. Status saved to:", XAI_DIR / "lime_run_status.csv")


## Final Validation Checklist

In [ ]:
# ==============================================================================
# Cell 24 : Modeling Validation Checklist
# ==============================================================================
# Mirrors the PASS/FAIL checklist style used in 01_Preprocessing_local.ipynb.
# Run this LAST, after the model, stats, and XAI cells you care about.
# ==============================================================================

import json as _json

print("=" * 80)
print("FlowGraphFormer Modeling Validation")
print("=" * 80)

modeling_results = {}

def check(name, condition):

    status = "PASS" if condition else "FAIL"

    modeling_results[name] = bool(condition)

    print(f"{name:<45}{status:>35}")

# ------------------------------------------------------------------
# Config / data
# ------------------------------------------------------------------
check("Run config saved", (MODEL_REPORT_DIR / "run_config.json").exists())

check("Data leakage audit saved", (TABLE_DIR / "data_leakage_audit.csv").exists())

check("Fold indices saved", (MODEL_REPORT_DIR / "fold_indices.pkl").exists())

# ------------------------------------------------------------------
# Per-model artifacts
# ------------------------------------------------------------------
model_names = ["catboost", "fttransformer", "tabm", "tabgt", "gnnformer", "graphtransformer", "tabpfn"]

for name in model_names:

    check(f"CV metrics : {name}", (TABLE_DIR / f"{name}_cv_fold_metrics.csv").exists())

    check(f"Test metrics : {name}", (TABLE_DIR / f"{name}_test_metrics.csv").exists())

# ------------------------------------------------------------------
# Comparison / stats / XAI
# ------------------------------------------------------------------
check("All-model CV summary", (TABLE_DIR / "all_models_cv_summary.csv").exists())

check("All-model test summary", (TABLE_DIR / "all_models_test_summary.csv").exists())

check("Paired t-test results", (STATS_DIR / "paired_ttest_results.csv").exists())

check("McNemar test results", (STATS_DIR / "mcnemar_test_results.csv").exists())

check("SHAP run status (all models)", (XAI_DIR / "shap_run_status.csv").exists())

check("Permutation importance run status (all models)", (XAI_DIR / "permutation_importance_run_status.csv").exists())

check("LIME run status (all models)", (XAI_DIR / "lime_run_status.csv").exists())

for _xai_model_key in CONFIG.get("xai_models", []):
    check(f"SHAP bar : {_xai_model_key}", (XAI_DIR / f"shap_summary_bar_{_xai_model_key}.{CONFIG['figure_format']}").exists())
    check(f"Permutation importance : {_xai_model_key}", (XAI_DIR / f"permutation_importance_{_xai_model_key}.csv").exists())
    check(f"LIME explanation : {_xai_model_key}", (XAI_DIR / f"lime_instance_{CONFIG['lime_instance_index']}_{_xai_model_key}.html").exists())

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
passed = sum(modeling_results.values())

failed = len(modeling_results) - passed

print()
print("=" * 80)
print(f"Total Checks : {len(modeling_results)}")
print(f"Passed       : {passed}")
print(f"Failed       : {failed}")
print("=" * 80)

results_df = pd.DataFrame({
    "Check": list(modeling_results.keys()),
    "Status": ["PASS" if v else "FAIL" for v in modeling_results.values()],
})

results_df.to_csv(MODEL_REPORT_DIR / "modeling_validation_results.csv", index=False)

with open(MODEL_REPORT_DIR / "modeling_validation_results.json", "w") as f:

    _json.dump(modeling_results, f, indent=4)

print()
print("Saved:", MODEL_REPORT_DIR / "modeling_validation_results.csv")
print("Saved:", MODEL_REPORT_DIR / "modeling_validation_results.json")
